# L01 — 생성 모델의 기초: likelihood, KL, ELBO, 그리고 평가

이 파일은 **jupytext "percent" 형식**의 Python 파일입니다.

- `# %% [markdown]` 셀 = 설명 (Markdown + LaTeX)
- `# %%` 셀 = 실행 코드

```bash
cd tutorial && bash ../tools/gpu_slot.sh S ../.venv/bin/python L01_GenBasics.py           # 전체 실행 (~25 분, checkpoint cache 후 재실행은 수 분)
cd tutorial && QUICK=1 bash ../tools/gpu_slot.sh S ../.venv/bin/python L01_GenBasics.py   # 짧은 실행 (~5 분, runs/_quick/)
python ../tools/py2ipynb.py L01_GenBasics.py                                              # -> L01_GenBasics.ipynb
```

**생성 모델 (generative model)** 은 데이터 분포 $p_{\text{data}}(x)$ 를 흉내 내는 분포 $p_\theta(x)$ 입니다.
우리가 원하는 일은 두 가지 중 하나 (또는 둘 다) 입니다.

1. **Density estimation**: 주어진 $x$ 에 대해 $\log p_\theta(x)$ 를 계산 (이상 탐지, 압축, likelihood 비교)
2. **Sampling**: $x\sim p_\theta$ 를 빠르게 뽑기 (이미지/동작 생성)

모든 모델 family 는 이 두 능력 사이에서 trade-off 를 고릅니다. 이번 강의는 이후 12 개 강의 (VAE → diffusion → flow matching → DiT → robot policy) 의 공통 언어를 만듭니다.

| 강의 내용 | 이 코드에서 |
|---|---|
| 2D toy 데이터 4 종 (8 Gaussians, two moons, checkerboard, Swiss roll) | §1 |
| Maximum likelihood = forward KL, forward vs reverse KL (mode covering / seeking) | §2 |
| Latent variable model 과 ELBO 유도, EM 으로 GMM 학습 | §3 |
| Autoregressive model (이산화 2D, chain rule) | §4 |
| Normalizing flow (RealNVP affine coupling, exact log-likelihood) | §5 |
| GAN (implicit model, mode collapse) | §6 |
| 모델 비교: NLL, MMD, precision / recall (Kynkäänniemi et al., NeurIPS 2019) | §7 |
| CIFAR-10 FID: real-vs-real, blur / noise 민감도, sample 수 bias, KID, bits/dim baseline | §8 |
| 생성 모델 family 지도 | §9 |

## 0. Setup

- Pretrained 생성 모델은 쓰지 않습니다. FID 용 Inception-v3 weight 만 `~/.cache/torch/hub/checkpoints` 에서 **offline** 으로 읽습니다.
- GPU 는 다른 강의와 공유합니다 (`tools/gpu_slot.sh`). 학습한 toy model 은 `runs/L01_GenBasics/` 에 저장되고, 다시 실행하면 불러옵니다.
- `QUICK=1`: 학습 step 과 sample 수를 줄인 짧은 실행. 결과는 `runs/_quick/L01_GenBasics/` 로만 갑니다.

In [ ]:
# Colab: install packages not preinstalled (torchmetrics FID needs torch-fidelity)
!pip install -q torchmetrics torch-fidelity imageio

In [ ]:
import os
import sys
import json
import math
import time
import copy

# torch 2.14 routes some small batched matmuls to Triton kernels that need a C compiler + Python headers at run time;
# they are missing on this machine, so keep the regular ATen kernels.
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

QUICK = os.environ.get("QUICK", "0") == "1"
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False      # exact fp32 everywhere: NLLs and FIDs are compared to 3 digits
torch.backends.cudnn.allow_tf32 = False

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
DATA = os.path.join(HERE, "data")
FIG = os.path.join(HERE, "figures")
RES = os.path.join(HERE, "results")
MEDIA = os.path.join(HERE, "media")
RUNS = os.path.join(HERE, "runs", "L01_GenBasics")
if QUICK:  # QUICK runs go to runs/_quick/ so they never overwrite the full-run figures / numbers quoted on the slides
    _QDIR = os.path.join(HERE, "runs", "_quick", "L01_GenBasics")
    FIG, RES, MEDIA, RUNS = (os.path.join(_QDIR, d) for d in ("figures", "results", "media", "runs"))
for d in (FIG, RES, MEDIA, RUNS):
    os.makedirs(d, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
DEV_NAME = torch.cuda.get_device_name(0) if device == "cuda" else "cpu"
print("device:", device, "|", DEV_NAME, "| torch", torch.__version__, "| QUICK =", QUICK)

plt.rcParams.update({"font.size": 14, "axes.titlesize": 15, "axes.labelsize": 14, "xtick.labelsize": 12,
                     "ytick.labelsize": 12, "legend.fontsize": 13, "figure.facecolor": "white",
                     "savefig.facecolor": "white"})

RES_PATH = os.path.join(RES, "L01_GenBasics.json")
RESULTS = {"device": DEV_NAME, "dtype": "fp32 (TF32 off)", "quick": QUICK, "torch": torch.__version__}
T_START = time.time()


def savefig(fig, name):
    path = os.path.join(FIG, f"L01_{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", os.path.relpath(path, HERE))


def save_results():
    with open(RES_PATH, "w") as f:
        json.dump(RESULTS, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))


def rnd(x, k=4):
    return float(round(float(x), k))


CIFAR = os.path.join(DATA, "cifar10_uint8.npz")
if not os.path.exists(CIFAR):  # Colab: download CIFAR-10 with torchvision and cache it as a uint8 npz
    import torchvision
    os.makedirs(DATA, exist_ok=True)
    _tr = torchvision.datasets.CIFAR10(DATA, train=True, download=True)
    _te = torchvision.datasets.CIFAR10(DATA, train=False, download=True)
    np.savez(CIFAR, train_x=_tr.data, train_y=np.array(_tr.targets), test_x=_te.data,
             test_y=np.array(_te.targets), classes=np.array(_tr.classes))
    print("built", CIFAR)

## 1. 2D toy 데이터: 눈으로 볼 수 있는 분포

이미지 생성 모델의 실패 (mode 누락, 흐릿함, 경계 밖 sample) 는 고차원에서는 보이지 않습니다. 2D 에서는 **density 와 sample 을 직접 그릴 수 있으므로**
모든 개념을 먼저 2D 에서 확인합니다. 네 가지 분포는 서로 다른 어려움을 줍니다.

| 데이터 | 어려움 | 정답 density |
|---|---|---|
| **8 Gaussians** (반지름 2.5, $\sigma=0.15$) | 멀리 떨어진 8 개 mode — mode 누락이 바로 보임 | 해석적: $\frac18\sum_k\mathcal N(x;c_k,\sigma^2 I)$ |
| **Two moons** | 휘어진 1D manifold + 잡음 | 없음 |
| **Checkerboard** | 날카로운 경계, 서로 붙어 있지 않은 8 개 정사각형 | 해석적: 정사각형 안에서 균일 $1/(8a^2)$ |
| **Swiss roll** | 나선형으로 감긴 manifold, 가까운 두 팔 | 없음 |

모든 데이터는 $[-4,4]^2$ 안에 들어오도록 scale 했습니다. Train 50k / test 20k 를 고정 seed 로 한 번 뽑아 모든 모델이 같은 데이터를 씁니다.

In [ ]:
GAUSS_R, GAUSS_S = 2.5, 0.15
GAUSS_C = torch.tensor([[GAUSS_R * math.cos(2 * math.pi * k / 8), GAUSS_R * math.sin(2 * math.pi * k / 8)]
                        for k in range(8)], device=device)
CB_A = 1.75  # checkerboard square side


def sample_toy(name, n, gen):
    """Draw n samples (n, 2) from one of the four toy distributions; gen = torch.Generator on `device`."""
    U = lambda *s: torch.rand(*s, device=device, generator=gen)
    N = lambda *s: torch.randn(*s, device=device, generator=gen)
    if name == "8gauss":
        k = torch.randint(0, 8, (n,), device=device, generator=gen)
        return GAUSS_C[k] + GAUSS_S * N(n, 2)
    if name == "moons":
        t = math.pi * U(n)
        up = U(n) < 0.5
        x = torch.where(up, torch.cos(t), 1 - torch.cos(t))
        y = torch.where(up, torch.sin(t), 0.5 - torch.sin(t))
        p = torch.stack([x - 0.5, y - 0.25], 1) * 2.2
        return p + 0.12 * N(n, 2)
    if name == "checkerboard":                       # FFJORD-style board, 8 squares of side CB_A
        x1 = U(n) * 4 - 2
        x2 = U(n) - torch.randint(0, 2, (n,), device=device, generator=gen).float() * 2
        x2 = x2 + torch.floor(x1) % 2
        return torch.stack([x1, x2], 1) * CB_A
    if name == "swissroll":
        t = 1.5 * math.pi * (1 + 2 * U(n))
        p = torch.stack([t * torch.cos(t), t * torch.sin(t)], 1) / 4.0
        return p + 0.1 * N(n, 2)
    raise ValueError(name)


def true_logp(name, x):
    """Exact log density where it exists in closed form (8 Gaussians, checkerboard), else None."""
    if name == "8gauss":
        d2 = ((x[:, None, :] - GAUSS_C[None]) ** 2).sum(-1)
        lc = -d2 / (2 * GAUSS_S ** 2) - math.log(2 * math.pi * GAUSS_S ** 2)
        return torch.logsumexp(lc, 1) - math.log(8)
    if name == "checkerboard":
        u = x / CB_A
        inside = (u[:, 0].abs() <= 2) & (u[:, 1].abs() <= 2) & \
                 ((torch.floor(u[:, 0]) + torch.floor(u[:, 1])) % 2 == 0)
        return torch.where(inside, torch.full_like(u[:, 0], -math.log(8 * CB_A ** 2)),
                           torch.full_like(u[:, 0], -float("inf")))
    return None


TOYS = ["8gauss", "moons", "checkerboard", "swissroll"]
TOY_TITLE = {"8gauss": "8 Gaussians", "moons": "Two moons", "checkerboard": "Checkerboard", "swissroll": "Swiss roll"}
N_TRAIN, N_TEST = (50_000, 20_000) if not QUICK else (20_000, 5_000)
g = torch.Generator(device=device).manual_seed(SEED)
TRAIN = {k: sample_toy(k, N_TRAIN, g) for k in TOYS}
TEST = {k: sample_toy(k, N_TEST, g) for k in TOYS}
LIM = 4.0
RESULTS["toy"] = {"n_train": N_TRAIN, "n_test": N_TEST, "true_test_nll_nats": {}}
for k in TOYS:
    lp = true_logp(k, TEST[k])
    if lp is not None:  # entropy estimate = the best possible test NLL
        RESULTS["toy"]["true_test_nll_nats"][k] = rnd(-lp.mean())
print("true test NLL (= entropy estimate):", RESULTS["toy"]["true_test_nll_nats"])

fig, axs = plt.subplots(1, 4, figsize=(16, 4.3))
for ax, k in zip(axs, TOYS):
    x = TRAIN[k][:5000].cpu()
    ax.scatter(x[:, 0], x[:, 1], s=1.5, alpha=0.5, color="#2774B8")
    ax.set_title(TOY_TITLE[k])
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_xticks([-4, 0, 4]); ax.set_yticks([-4, 0, 4])
fig.suptitle("Four 2D toy distributions (5,000 training samples each)", y=1.02)
savefig(fig, "toy_datasets")

**관찰 포인트**: 8 Gaussians 와 checkerboard 는 정답 density 를 알기 때문에, test NLL 의 **하한** (= 데이터의 differential entropy $H(p_{\text{data}})$ 의 Monte-Carlo 추정) 을
계산할 수 있습니다. 위에서 출력된 값 이하로는 어떤 모델도 내려갈 수 없습니다 (§2 에서 이유를 유도).

## 2. Maximum likelihood = forward KL, 그리고 reverse KL

### 2.1 유도

데이터 분포 $p=p_{\text{data}}$, 모델 $q=p_\theta$. **Forward KL** 은

$$
\mathrm{KL}(p\,\|\,p_\theta)=\mathbb E_{x\sim p}\big[\log p(x)-\log p_\theta(x)\big]
=\underbrace{-H(p)}_{\theta\text{ 와 무관}}\;-\;\mathbb E_{x\sim p}\big[\log p_\theta(x)\big].
$$

따라서

$$
\arg\min_\theta \mathrm{KL}(p\,\|\,p_\theta)=\arg\max_\theta \mathbb E_{p}[\log p_\theta(x)]
\;\approx\;\arg\max_\theta \frac1N\sum_{n=1}^N\log p_\theta(x_n)\quad(\text{MLE}).
$$

즉 **test NLL $=-\mathbb E_p\log p_\theta = H(p)+\mathrm{KL}(p\|p_\theta)\ge H(p)$**. NLL 과 entropy 의 차이가 곧 forward KL 입니다.
MLE 는 $p$ 에서 뽑은 sample 만 필요하고 $p$ 의 density 는 필요 없습니다 — 그래서 데이터만 있는 생성 모델링의 기본 목적함수가 됩니다.

**Reverse KL** $\mathrm{KL}(p_\theta\|p)=\mathbb E_{x\sim p_\theta}[\log p_\theta(x)-\log p(x)]$ 는 $p$ 의 density 를 알아야 합니다
(variational inference, 에너지 기반 target, RL policy 등). 두 방향은 실패 방식이 다릅니다.

- Forward: $p(x)>0$ 인데 $p_\theta(x)\to0$ 이면 $-\log p_\theta\to\infty$ → **모든 mode 를 덮어야 함 (mode covering, zero-avoiding)**. 대가: mode 사이 빈 공간에도 확률을 둠.
- Reverse: $p_\theta(x)>0$ 인데 $p(x)\to0$ 이면 $-\log p\to\infty$ → **데이터가 없는 곳에 질량을 두면 안 됨 (mode seeking, zero-forcing)**. 대가: 한 mode 만 고르고 나머지를 버려도 벌점이 작음.

### 2.2 Gaussian $q$ 의 forward KL 해 = moment matching

$q=\mathcal N(\mu,\Sigma)$ 일 때 $-\mathbb E_p\log q=\tfrac12\mathbb E_p[(x-\mu)^\top\Sigma^{-1}(x-\mu)]+\tfrac12\log\det\Sigma+\text{const}$.

- $\nabla_\mu$: $\Sigma^{-1}(\mu-\mathbb E_p x)=0\Rightarrow \mu^\star=\mathbb E_p[x]$
- $\nabla_{\Sigma^{-1}}$: $\tfrac12\mathbb E_p[(x-\mu)(x-\mu)^\top]-\tfrac12\Sigma=0\Rightarrow \Sigma^\star=\mathrm{Cov}_p[x]$

닫힌 해가 있습니다. Reverse KL 은 닫힌 해가 없으므로 **reparameterization** $x=\mu+L\varepsilon,\ \varepsilon\sim\mathcal N(0,I)$ 로
$\nabla_{\mu,L}\,\mathbb E_\varepsilon[\log q(\mu+L\varepsilon)-\log p(\mu+L\varepsilon)]$ 를 Monte-Carlo 로 추정해 SGD 합니다 (L02 VAE 의 핵심 trick 을 미리 봄).
이때 $\log q$ 항의 기댓값은 entropy $-H(q)=-\tfrac12\log\det(2\pi e\,\Sigma)$ 로 닫힌 형태입니다.

Target 은 두 개입니다: (a) 무게가 다른 3-성분 Gaussian mixture, (b) 8 Gaussians ring. 각각에 대해 forward fit 1 개와 reverse fit 8 개 (서로 다른 초기값) 를 비교합니다.

In [ ]:
class GaussMix:
    """A fixed 2D Gaussian mixture with exact log-density and sampling (the target p)."""

    def __init__(self, w, mu, cov):
        self.w = torch.tensor(w, device=device, dtype=torch.float32)
        self.mu = torch.tensor(mu, device=device, dtype=torch.float32)
        self.cov = torch.tensor(cov, device=device, dtype=torch.float32)
        self.L = torch.linalg.cholesky(self.cov)
        self.P = torch.linalg.inv(self.cov)
        self.logdet = torch.logdet(self.cov)

    def log_prob(self, x):
        d = x[:, None, :] - self.mu[None]
        m = torch.einsum("nki,kij,nkj->nk", d, self.P, d)
        lc = -0.5 * m - 0.5 * self.logdet - math.log(2 * math.pi) + torch.log(self.w)
        return torch.logsumexp(lc, 1)

    def sample(self, n, gen):
        k = torch.multinomial(self.w, n, replacement=True, generator=gen)
        e = torch.randn(n, 2, device=device, generator=gen)
        return self.mu[k] + torch.einsum("nij,nj->ni", self.L[k], e)


def gauss_logpdf(x, mu, L):
    """log N(x; mu, L L^T) for a single full-covariance Gaussian."""
    d = torch.linalg.solve_triangular(L, (x - mu).T, upper=False).T
    return -0.5 * (d ** 2).sum(1) - torch.log(torch.diagonal(L)).sum() - math.log(2 * math.pi)


TARGETS = {
    "mix3": GaussMix([0.45, 0.35, 0.20], [[-2.0, -0.8], [2.0, -0.8], [0.0, 2.2]],
                     [[[0.5, 0.25], [0.25, 0.4]], [[0.3, -0.1], [-0.1, 0.6]], [[0.6, 0.0], [0.0, 0.15]]]),
    "ring8": GaussMix([1 / 8] * 8, GAUSS_C.cpu().tolist(), [[[GAUSS_S ** 2, 0], [0, GAUSS_S ** 2]]] * 8),
}


def fit_reverse_kl(target, init_mu, steps=3000, n=1024, lr=0.03, seed=0):
    """min_q KL(q || p) for q = N(mu, L L^T): reparameterized Monte-Carlo gradient."""
    gen = torch.Generator(device=device).manual_seed(seed)
    mu = torch.tensor(init_mu, device=device, dtype=torch.float32).requires_grad_(True)
    raw = torch.zeros(3, device=device, requires_grad=True)          # L = [[e^a, 0], [c, e^b]]
    opt = torch.optim.Adam([mu, raw], lr=lr)
    for it in range(steps):
        L = torch.stack([torch.stack([raw[0].exp(), torch.zeros((), device=device)]),
                         torch.stack([raw[2], raw[1].exp()])])
        eps = torch.randn(n, 2, device=device, generator=gen)
        x = mu + eps @ L.T                                            # reparameterization trick
        neg_entropy = -(torch.log(torch.diagonal(L)).sum() + math.log(2 * math.pi * math.e))  # E_q log q, exact
        loss = neg_entropy - target.log_prob(x).mean()                # = KL(q||p) up to nothing (exact entropy)
        opt.zero_grad(); loss.backward(); opt.step()
    L = torch.stack([torch.stack([raw[0].exp(), torch.zeros((), device=device)]), torch.stack([raw[2], raw[1].exp()])])
    return mu.detach(), L.detach()


def kl_both(target, mu, L, n=200_000, seed=1):
    """Monte-Carlo KL(p||q) and KL(q||p) (nats)."""
    gen = torch.Generator(device=device).manual_seed(seed)
    xp = target.sample(n, gen)
    fwd = (target.log_prob(xp) - gauss_logpdf(xp, mu, L)).mean()
    xq = mu + torch.randn(n, 2, device=device, generator=gen) @ L.T
    rev = (gauss_logpdf(xq, mu, L) - target.log_prob(xq)).mean()
    return fwd.item(), rev.item()


KL_STEPS = 3000 if not QUICK else 600
kl_res = {}
kl_fits = {}
gen = torch.Generator(device=device).manual_seed(SEED)
init_pts = (torch.rand(8, 2, device=device, generator=gen) * 6 - 3).tolist()
for tname, tgt in TARGETS.items():
    xs = tgt.sample(200_000, torch.Generator(device=device).manual_seed(2))
    mu_f = xs.mean(0)                                                  # forward KL: moment matching
    L_f = torch.linalg.cholesky(torch.cov(xs.T))
    rev = [fit_reverse_kl(tgt, ip, steps=KL_STEPS, seed=i) for i, ip in enumerate(init_pts)]
    f_kl = kl_both(tgt, mu_f, L_f)
    r_kl = [kl_both(tgt, m, L) for m, L in rev]
    kl_fits[tname] = (mu_f, L_f, rev)
    kl_res[tname] = {"forward_fit": {"KL_p_q": rnd(f_kl[0]), "KL_q_p": rnd(f_kl[1])},
                     "reverse_fits": [{"init": [rnd(a, 2) for a in ip], "mu": [rnd(a, 3) for a in m.tolist()],
                                       "KL_p_q": rnd(k[0]), "KL_q_p": rnd(k[1])}
                                      for ip, (m, _), k in zip(init_pts, rev, r_kl)]}
    best = min(r_kl, key=lambda t: t[1])
    print(f"{tname}: forward fit  KL(p||q)={f_kl[0]:.3f}  KL(q||p)={f_kl[1]:.3f}")
    print(f"{tname}: best reverse KL(p||q)={best[0]:.3f}  KL(q||p)={best[1]:.3f};  all reverse KL(q||p):",
          [round(k[1], 3) for k in r_kl])

# Gradient check: SGD on the forward KL (= MLE on samples) reaches the same moments as the closed form
tgt = TARGETS["mix3"]
xs = tgt.sample(50_000, torch.Generator(device=device).manual_seed(3))
mu = torch.zeros(2, device=device, requires_grad=True); raw = torch.zeros(3, device=device, requires_grad=True)
opt = torch.optim.Adam([mu, raw], lr=0.05)
for it in range(1500 if not QUICK else 300):
    L = torch.stack([torch.stack([raw[0].exp(), torch.zeros((), device=device)]), torch.stack([raw[2], raw[1].exp()])])
    loss = -gauss_logpdf(xs, mu, L).mean()
    opt.zero_grad(); loss.backward(); opt.step()
mle_gap = (mu.detach() - xs.mean(0)).abs().max().item()
kl_res["mle_sgd_vs_moments_max_abs_diff_mu"] = rnd(mle_gap, 5)
print(f"MLE by SGD vs sample mean: max |diff| = {mle_gap:.2e}")
RESULTS["kl"] = kl_res
save_results()

In [ ]:
def ellipse(ax, mu, L, color, lw=2.5, ls="-", nsig=(1, 2)):
    t = torch.linspace(0, 2 * math.pi, 200, device=device)
    for s in nsig:
        c = mu[None] + s * torch.stack([t.cos(), t.sin()], 1) @ L.T
        c = c.cpu()
        ax.plot(c[:, 0], c[:, 1], color=color, lw=lw if s == 1 else lw * 0.6, ls=ls)


def density_grid(logp_fn, lim=LIM, n=300):
    xs = torch.linspace(-lim, lim, n, device=device)
    X, Y = torch.meshgrid(xs, xs, indexing="xy")
    P = torch.stack([X.reshape(-1), Y.reshape(-1)], 1)
    with torch.no_grad():
        lp = torch.cat([logp_fn(P[i:i + 40000]) for i in range(0, len(P), 40000)])
    return lp.reshape(n, n).exp().cpu().numpy()


fig, axs = plt.subplots(1, 2, figsize=(14, 6.6))
for ax, (tname, tgt) in zip(axs, TARGETS.items()):
    D = density_grid(tgt.log_prob)
    ax.imshow(D, extent=[-LIM, LIM, -LIM, LIM], origin="lower", cmap="Greys", vmax=D.max() * 0.8)
    mu_f, L_f, rev = kl_fits[tname]
    ellipse(ax, mu_f, L_f, "#2774B8", lw=3.5)
    for m, L in rev:
        ellipse(ax, m, L, "#E97820", lw=2.2, ls="-")
    ax.plot([], [], color="#2774B8", lw=3.5, label=f"forward KL (MLE): KL(p||q)={kl_res[tname]['forward_fit']['KL_p_q']:.2f}")
    rbest = min(kl_res[tname]["reverse_fits"], key=lambda r: r["KL_q_p"])
    ax.plot([], [], color="#E97820", lw=2.2, label=f"reverse KL, 8 inits: best KL(q||p)={rbest['KL_q_p']:.2f}")
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal")
    ax.set_title({"mix3": "(a) 3-component mixture (weights .45/.35/.20)", "ring8": "(b) 8 Gaussians ring"}[tname])
    ax.legend(loc="lower center", fontsize=12, framealpha=0.95)
fig.suptitle("Fitting a single Gaussian q to a mixture p: 1σ / 2σ ellipses", y=0.98)
fig.tight_layout()
savefig(fig, "fwd_rev_kl")

**관찰 포인트**

- 파란 forward fit 은 모든 mode 를 감싸는 넓은 Gaussian 입니다 — 그 중심 (mode 사이) 에는 데이터가 거의 없는데도 density 가 가장 높습니다.
  이미지 모델에서 이것이 **흐릿한 평균 이미지** 로 나타납니다 (L02 VAE).
- 주황 reverse fit 은 초기값에 따라 **서로 다른 mode 하나** 에 붙습니다. Reverse KL 은 non-convex 이고 local optimum 이 여러 개 — 각 해는 "그럴듯하지만 다양성이 없는" sample 을 냅니다 (GAN 의 mode collapse 와 같은 증상, §6).
- 표의 숫자: forward fit 은 KL(p‖q) 가 작지만 KL(q‖p) 가 크고, reverse fit 은 그 반대입니다. **어느 쪽이 "좋은 모델" 인가는 어떤 divergence 로 재느냐에 달려 있습니다.**

## 3. Latent variable model, ELBO, EM

### 3.1 Latent variable model

관측되지 않은 변수 $z$ 를 도입해 복잡한 $p_\theta(x)$ 를 간단한 조각으로 만듭니다:

$$
p_\theta(x)=\int p_\theta(x\mid z)\,p(z)\,dz\qquad\Big(\text{GMM: }z\in\{1..K\},\ p(z=k)=\pi_k,\ p(x\mid z=k)=\mathcal N(x;\mu_k,\Sigma_k)\Big).
$$

VAE (L02) 는 $z\in\mathbb R^d$, $p(z)=\mathcal N(0,I)$, $p_\theta(x\mid z)$ = neural decoder 인 같은 구조이고, diffusion (L04) 은 $z=x_{1:T}$ 인 깊은 latent variable model 입니다.
문제는 $z$ 에 대한 적분/합 때문에 $\log p_\theta(x)$ 와 posterior $p_\theta(z\mid x)$ 가 일반적으로 계산 불가능하다는 것입니다.

### 3.2 ELBO 유도 (두 가지 방법)

임의의 분포 $q(z)$ ($q(z)>0$ where $p(x,z)>0$) 를 도입합니다.

**(i) Jensen 부등식**: $\log$ 는 concave 이므로

$$
\log p_\theta(x)=\log\int q(z)\frac{p_\theta(x,z)}{q(z)}dz=\log\mathbb E_{q}\Big[\frac{p_\theta(x,z)}{q(z)}\Big]
\;\ge\;\mathbb E_{q}\Big[\log\frac{p_\theta(x,z)}{q(z)}\Big]=:\mathcal L(q,\theta)\quad(\text{ELBO}).
$$

**(ii) 정확한 분해** (gap 이 무엇인지 보여 줌): $p_\theta(x)=p_\theta(x,z)/p_\theta(z\mid x)$ 는 모든 $z$ 에서 성립하므로 $q$ 에 대한 기댓값을 취하면

$$
\log p_\theta(x)=\mathbb E_q\Big[\log\frac{p_\theta(x,z)}{p_\theta(z\mid x)}\Big]
=\mathbb E_q\Big[\log\frac{p_\theta(x,z)}{q(z)}\Big]+\mathbb E_q\Big[\log\frac{q(z)}{p_\theta(z\mid x)}\Big]
=\mathcal L(q,\theta)+\mathrm{KL}\big(q(z)\,\|\,p_\theta(z\mid x)\big).
$$

KL $\ge0$ 이므로 (i) 과 같은 부등식이 나오고, **등호 조건은 $q(z)=p_\theta(z\mid x)$** 입니다.
또 $p_\theta(x,z)=p_\theta(x\mid z)p(z)$ 를 넣으면 VAE 에서 쓰는 형태가 됩니다:

$$
\mathcal L(q,\theta)=\underbrace{\mathbb E_{q}[\log p_\theta(x\mid z)]}_{\text{reconstruction}}-\underbrace{\mathrm{KL}\big(q(z)\,\|\,p(z)\big)}_{\text{prior 와의 거리}}.
$$

### 3.3 EM = ELBO 의 coordinate ascent

- **E-step**: $\theta$ 고정, $q$ 에 대해 최대화 → $q^{(t)}(z)=p_{\theta^{(t)}}(z\mid x)$ (KL = 0, bound 가 tight: $\mathcal L=\log p$)
- **M-step**: $q$ 고정, $\theta$ 에 대해 최대화 → $\theta^{(t+1)}=\arg\max_\theta \mathbb E_{q^{(t)}}[\log p_\theta(x,z)]$

$$
\log p_{\theta^{(t)}}(x)=\mathcal L(q^{(t)},\theta^{(t)})\le\mathcal L(q^{(t)},\theta^{(t+1)})\le\log p_{\theta^{(t+1)}}(x)
$$

이므로 log-likelihood 는 **단조 증가** 합니다 (아래에서 수치로 확인). GMM 에서 responsibility $r_{nk}=q(z_n=k)$ 를 쓰면 M-step 은 닫힌 형태입니다
($\sum_k\pi_k=1$ 은 Lagrange multiplier 로):

$$
N_k=\sum_n r_{nk},\quad \pi_k=\frac{N_k}{N},\quad \mu_k=\frac{1}{N_k}\sum_n r_{nk}x_n,\quad
\Sigma_k=\frac{1}{N_k}\sum_n r_{nk}(x_n-\mu_k)(x_n-\mu_k)^\top .
$$

VAE 는 $p_\theta(z\mid x)$ 를 계산할 수 없어서 E-step 을 정확히 못 하고, 대신 encoder $q_\phi(z\mid x)$ 로 근사해 $\phi,\theta$ 를 함께 SGD 합니다 (amortized VI, L02).

In [ ]:
def gmm_log_joint(x, logpi, mu, cov):
    """log pi_k + log N(x_n; mu_k, Sigma_k): (N, K)."""
    P = torch.linalg.inv(cov)
    d = x[:, None, :] - mu[None]
    m = torch.einsum("nki,kij,nkj->nk", d, P, d)
    return logpi[None] - 0.5 * m - 0.5 * torch.logdet(cov)[None] - 0.5 * x.shape[1] * math.log(2 * math.pi)


def kmeanspp(x, K, gen):
    idx = [torch.randint(0, len(x), (1,), device=x.device, generator=gen).item()]
    d2 = ((x - x[idx[0]]) ** 2).sum(1)
    for _ in range(K - 1):
        j = torch.multinomial(d2 / d2.sum(), 1, generator=gen).item()
        idx.append(j)
        d2 = torch.minimum(d2, ((x - x[j]) ** 2).sum(1))
    return x[idx].clone()


def fit_gmm(x, K, iters=300, seed=0, reg=1e-4, init="kmeans++", tol=1e-7, dtype=torch.float64):
    """EM for a full-covariance GMM (float64 by default, for the exact monotonicity check).
    Returns params and the per-iteration (log-lik, ELBO after M-step)."""
    x = x.to(dtype)
    N, D = x.shape
    gen = torch.Generator(device=x.device).manual_seed(seed)
    mu = kmeanspp(x, K, gen) if init == "kmeans++" else x[torch.randperm(N, device=x.device, generator=gen)[:K]].clone()
    cov = torch.eye(D, device=x.device, dtype=x.dtype)[None].repeat(K, 1, 1) * x.var(0).mean()
    logpi = torch.full((K,), -math.log(K), device=x.device, dtype=x.dtype)
    hist_ll, hist_elbo = [], []
    for it in range(iters):
        lj = gmm_log_joint(x, logpi, mu, cov)
        ll = torch.logsumexp(lj, 1)
        hist_ll.append(ll.mean().item())                 # = ELBO(q_t, theta_t) because q_t is the exact posterior
        r = (lj - ll[:, None]).exp()                     # E-step: responsibilities r_nk = p(z=k | x_n)
        Nk = r.sum(0) + 1e-10                            # M-step (closed form)
        logpi = torch.log(Nk / N)
        mu = (r.T @ x) / Nk[:, None]
        d = x[:, None, :] - mu[None]
        cov = torch.einsum("nk,nki,nkj->kij", r, d, d) / Nk[:, None, None] + reg * torch.eye(D, device=x.device, dtype=x.dtype)
        lj2 = gmm_log_joint(x, logpi, mu, cov)          # ELBO(q_t, theta_{t+1}) = E_q[log p(x,z)] + H(q)
        hist_elbo.append((r * (lj2 - torch.log(r.clamp_min(1e-300)))).sum(1).mean().item())
        if it > 2 and abs(hist_ll[-1] - hist_ll[-2]) < tol:
            break
    return {"logpi": logpi, "mu": mu, "cov": cov, "ll": hist_ll, "elbo": hist_elbo}


def gmm_logp(params, x):
    return torch.logsumexp(gmm_log_joint(x.to(params["mu"].dtype), params["logpi"], params["mu"], params["cov"]), 1).float()


def gmm_sample(params, n, gen):
    k = torch.multinomial(params["logpi"].exp().float(), n, replacement=True, generator=gen)
    L = torch.linalg.cholesky(params["cov"]).float()
    return params["mu"].float()[k] + torch.einsum("nij,nj->ni", L[k], torch.randn(n, 2, device=device, generator=gen))


# EM on 8 Gaussians with K = 8, random init (shows the monotone ascent and a local optimum) vs k-means++ init
t0 = time.time()
em_rand = fit_gmm(TRAIN["8gauss"], 8, iters=300, seed=3, init="random")
em_kpp = fit_gmm(TRAIN["8gauss"], 8, iters=300, seed=3, init="kmeans++")
ll_seq = np.array(em_rand["ll"]); el_seq = np.array(em_rand["elbo"])
monotone = bool(np.all(np.diff(ll_seq) >= -1e-9) and np.all(el_seq[:-1] <= ll_seq[1:] + 1e-9) and np.all(ll_seq <= el_seq + 1e-9))
print(f"EM random init: {len(ll_seq)} it, final train log-lik {ll_seq[-1]:.4f}; k-means++: {em_kpp['ll'][-1]:.4f}; "
      f"chain ll_t <= ELBO(q_t,th_t+1) <= ll_t+1 holds: {monotone}  ({time.time() - t0:.1f}s)")

# Random-init restarts: how often does EM get stuck?
restarts = [fit_gmm(TRAIN["8gauss"], 8, iters=300, seed=s, init="random")["ll"][-1] for s in range(10)]
RESULTS["em"] = {"rand_init_iters": len(ll_seq), "rand_init_final_ll": rnd(ll_seq[-1]),
                 "kpp_final_ll": rnd(em_kpp["ll"][-1]), "kpp_iters": len(em_kpp["ll"]),
                 "monotone_chain_holds": monotone,
                 "random_restarts_final_ll": [rnd(v) for v in restarts],
                 "true_train_ll": rnd(true_logp("8gauss", TRAIN["8gauss"]).mean())}
print("10 random restarts, final train log-lik:", [round(v, 3) for v in restarts])

In [ ]:
# Figure: EM snapshots + the ELBO / log-likelihood staircase
snap_its = [0, 2, 8, len(ll_seq) - 1]
snap_params = {}
x = TRAIN["8gauss"]
for it_stop in snap_its:
    snap_params[it_stop] = fit_gmm(x, 8, iters=it_stop + 1, seed=3, init="random", tol=0) if it_stop > 0 else None


def draw_gmm(ax, P):
    for k in range(P["mu"].shape[0]):
        if P["logpi"][k].exp() < 1e-3:
            continue
        ellipse(ax, P["mu"][k].float(), torch.linalg.cholesky(P["cov"][k]).float(), "#E97820", lw=2.2, nsig=(2,))
        ax.plot(*P["mu"][k].float().cpu().numpy(), "x", color="#E97820", ms=8, mew=2.5)


fig = plt.figure(figsize=(17, 4.8))
gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 1, 1, 1.55], wspace=0.38)
xs_cpu = x[:4000].cpu()
for j, it_stop in enumerate(snap_its):
    ax = fig.add_subplot(gs[0, j])
    ax.scatter(xs_cpu[:, 0], xs_cpu[:, 1], s=1, alpha=0.3, color="#2774B8")
    if it_stop == 0:  # draw the initialization: one EM step with 0 iterations is the init itself
        gen0 = torch.Generator(device=device).manual_seed(3)
        mu0 = x.double()[torch.randperm(len(x), device=device, generator=gen0)[:8]]
        for k in range(8):
            ax.plot(*mu0[k].float().cpu().numpy(), "x", color="#E97820", ms=8, mew=2.5)
        ax.set_title("init (random points)")
    else:
        draw_gmm(ax, snap_params[it_stop])
        ax.set_title(f"after {it_stop + 1} EM steps")
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
ax = fig.add_subplot(gs[0, 4])
n_show = min(len(ll_seq), 60)
stair_x, stair_y = [], []
for t in range(n_show - 1):
    stair_x += [t, t + 0.5]; stair_y += [ll_seq[t], el_seq[t]]
ax.plot(np.arange(n_show), ll_seq[:n_show], "o", color="#203864", ms=4, label=r"$\log p_{\theta_t}(x)$ = ELBO after E-step")
ax.plot(np.arange(n_show - 1) + 0.5, el_seq[:n_show - 1], "s", color="#E97820", ms=4, label=r"ELBO$(q_t,\theta_{t+1})$ after M-step")
ax.plot(stair_x, stair_y, "-", color="#A5A5A5", lw=1)
ax.axhline(RESULTS["em"]["true_train_ll"], color="#3D8C54", ls="--", lw=2, label="true density")
ax.set_xlabel("EM iteration"); ax.set_ylabel("mean train log-lik (nats)")
ax.set_ylim(ll_seq[1] - 0.2, RESULTS["em"]["true_train_ll"] + 0.15)
ax.legend(fontsize=11, loc="lower right"); ax.set_title(f"monotone ascent (random init, K = 8)")
savefig(fig, "em_gmm")

**관찰 포인트**

- 점 (E-step 후 = log-likelihood) 과 사각형 (M-step 후 ELBO) 이 계단처럼 교대로 올라갑니다 — 위 부등식 사슬을 수치로 확인 (`monotone_chain_holds`).
- Random init 은 두 성분이 한 mode 를 나눠 갖고 다른 두 mode 를 한 성분이 덮는 **local optimum** 에 자주 빠집니다. 10 번의 restart 의 최종 log-lik 분포와
  k-means++ init 결과를 비교해 보세요. EM 은 likelihood 를 올리기만 할 뿐 global optimum 을 보장하지 않습니다.

In [ ]:
# Model size: test NLL vs number of components K on all four datasets (k-means++ init)
# float32 here (GPU float64 is slow); the sweep is cached in runs/ so reruns skip it
K_LIST = [1, 2, 4, 8, 16, 32, 64] if not QUICK else [1, 4, 16]
gmm_path = os.path.join(RUNS, "gmm_sweep.pt")
t0 = time.time()
if os.path.exists(gmm_path):
    ck = torch.load(gmm_path, map_location=device)
    gmm_nll, GMM_BEST = ck["nll"], ck["best"]
else:
    gmm_nll = {k: {"train": [], "test": []} for k in TOYS}
    GMM_BEST = {}
    for name in TOYS:
        for K in K_LIST:
            P = fit_gmm(TRAIN[name], K, iters=200 if not QUICK else 60, seed=0, tol=1e-6, dtype=torch.float32)
            gmm_nll[name]["train"].append(-gmm_logp(P, TRAIN[name]).mean().item())
            gmm_nll[name]["test"].append(-gmm_logp(P, TEST[name]).mean().item())
            if K == K_LIST[-1]:
                GMM_BEST[name] = {k: v for k, v in P.items() if k not in ("ll", "elbo")}
    torch.save({"nll": gmm_nll, "best": GMM_BEST}, gmm_path)
print(f"GMM K sweep: {time.time() - t0:.1f}s")
RESULTS["gmm"] = {"K": K_LIST, **{n: {"train_nll": [rnd(v) for v in gmm_nll[n]["train"]],
                                      "test_nll": [rnd(v) for v in gmm_nll[n]["test"]]} for n in TOYS}}
save_results()

fig, ax = plt.subplots(figsize=(8.5, 5))
cols = ["#2774B8", "#E97820", "#3D8C54", "#6D237C"]
for c, name in zip(cols, TOYS):
    ax.plot(K_LIST, gmm_nll[name]["test"], "o-", color=c, lw=2.5, label=TOY_TITLE[name])
    ax.plot(K_LIST, gmm_nll[name]["train"], ":", color=c, lw=1.8)
    if name in RESULTS["toy"]["true_test_nll_nats"]:
        ax.axhline(RESULTS["toy"]["true_test_nll_nats"][name], color=c, ls="--", lw=1.2)
ax.set_xscale("log", base=2); ax.set_xlabel("number of components K"); ax.set_ylabel("NLL (nats / sample)")
ax.set_title("GMM by EM: test NLL (solid), train (dotted), true entropy (dashed)")
ax.legend(); ax.grid(alpha=0.3)
savefig(fig, "gmm_nll_vs_k")

**관찰 포인트**: K 를 늘리면 NLL 이 계속 떨어집니다. 8 Gaussians 는 K = 8 이면 충분할 것 같지만 (seed 0 의 k-means++ init 으로는) 두 mode 를 한 성분이 덮는 local optimum 에 빠지고, 여분의 성분이 있는 K = 16 에서야 정답 entropy 에 도달한 뒤 평평해집니다 — 실전에서 성분 수를 넉넉히 두는 이유입니다.
Checkerboard 는 날카로운 경계를 Gaussian 으로 근사해야 해서 K = 64 에서도 하한과 차이가 남습니다 — 모델 family 의 **표현력 한계** 입니다.
50k sample 로는 2D 에서 K = 64 정도까지 train/test 차이가 거의 없습니다 (overfitting 은 고차원 문제).

## 4. Autoregressive model: chain rule + 이산화

모든 분포는 chain rule 로 정확히 분해됩니다 (근사가 아님):

$$
p(x_1,\dots,x_D)=\prod_{d=1}^{D}p(x_d\mid x_{<d}).
$$

각 조건부를 **categorical** 로 두면 (값을 $B$ 개 bin 으로 이산화) 학습은 단순한 cross-entropy, likelihood 는 정확, sampling 은 $D$ 번의 순차 호출입니다.
PixelCNN / PixelRNN (van den Oord et al., ICML 2016) 은 픽셀 값 256 개, GPT 는 token vocabulary 를 bin 으로 씁니다. 여기서는 $D=2$, $B=64$ bin ($[-4,4]$, 폭 $\Delta=0.125$):

$$
p_\theta(i_1,i_2)=\mathrm{softmax}(\mathbf a)_{i_1}\cdot\mathrm{softmax}\big(f_\theta(i_1)\big)_{i_2},\qquad f_\theta=\text{MLP}(\text{Embedding}(i_1)).
$$

**연속 density 와의 관계 (dequantization)**: bin 안에서 균일하다고 두면 $p(x)=P(i_1,i_2)/\Delta^2$, 따라서

$$
\mathrm{NLL}_{\text{cont}}=\mathrm{NLL}_{\text{disc}}+D\log\Delta .
$$

이미지의 **bits/dim** 도 같은 계산입니다: 8-bit 픽셀 $x\in\{0..255\}$ 를 $[0,1]$ 로 scale 하면 $\Delta=1/256$ 이고, 연속 모델 (flow, diffusion) 의 NLL 을
$\text{bpd}=\big(\mathrm{NLL}_{\text{cont}}-D\log\Delta\big)/(D\ln2)=\mathrm{NLL}_{\text{cont}}/(D\ln 2)+8$ 로 이산 bits 로 바꿉니다 (uniform dequantization; Theis et al., ICLR 2016).

$B$ 가 작으면 양자화 오차 (bin 안의 구조를 못 봄), 크면 bin 당 데이터가 부족합니다 — bin 수 sweep 으로 확인합니다.

In [ ]:
class AR2D(nn.Module):
    """p(i1) * p(i2 | i1) over B x B bins; p(i1) is a free logit vector, p(i2|i1) an MLP on an embedding of i1."""

    def __init__(self, B=64, emb=64, h=256):
        super().__init__()
        self.B = B
        self.logit1 = nn.Parameter(torch.zeros(B))
        self.emb = nn.Embedding(B, emb)
        self.net = nn.Sequential(nn.Linear(emb, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, B))

    def to_bins(self, x):
        return ((x + LIM) / (2 * LIM) * self.B).floor().long().clamp(0, self.B - 1)

    def nll_disc(self, x):                                # nats per sample (both dims)
        i = self.to_bins(x)
        l1 = F.cross_entropy(self.logit1[None].expand(len(i), -1), i[:, 0], reduction="none")
        l2 = F.cross_entropy(self.net(self.emb(i[:, 0])), i[:, 1], reduction="none")
        return l1 + l2

    def log_prob(self, x):                                # continuous density, uniform within a bin
        delta = 2 * LIM / self.B
        return -self.nll_disc(x) - 2 * math.log(delta)

    @torch.no_grad()
    def sample(self, n, gen):
        delta = 2 * LIM / self.B
        i1 = torch.multinomial(self.logit1.softmax(0), n, replacement=True, generator=gen)
        i2 = torch.multinomial(self.net(self.emb(i1)).softmax(1), 1, generator=gen)[:, 0]   # sequential: x2 after x1
        i = torch.stack([i1, i2], 1).float()
        return -LIM + (i + torch.rand(n, 2, device=device, generator=gen)) * delta


def train_nn(model, loss_fn, x, steps, bs=2048, lr=1e-3, seed=0, log_every=0):
    """Generic minibatch Adam + cosine schedule on a tensor dataset already on the GPU."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    gen = torch.Generator(device=device).manual_seed(seed)
    hist = []
    for it in range(steps):
        idx = torch.randint(0, len(x), (bs,), device=device, generator=gen)
        loss = loss_fn(model, x[idx])
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); sched.step()
        if log_every and it % log_every == 0:
            hist.append((it, loss.item()))
    return hist


def cached(name, build, train):
    """Load a trained model from runs/ or train it (and save)."""
    path = os.path.join(RUNS, f"{name}.pt")
    model = build().to(device)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        return model, None
    t0 = time.time()
    hist = train(model)
    torch.save(model.state_dict(), path)
    return model, {"train_s": time.time() - t0, "hist": hist}


AR_STEPS = 4000 if not QUICK else 600
AR_MODELS, ar_info = {}, {}
for name in TOYS:
    m, info = cached(f"ar64_{name}", lambda: AR2D(64),
                     lambda mdl, name=name: train_nn(mdl, lambda md, xb: md.nll_disc(xb).mean(), TRAIN[name], AR_STEPS))
    m.eval(); AR_MODELS[name] = m
    with torch.no_grad():
        ar_info[name] = {"test_nll": rnd(-m.log_prob(TEST[name]).mean()), "train_s": rnd(info["train_s"], 1) if info else None}
    print(f"AR B=64 {name}: test NLL {ar_info[name]['test_nll']:.3f} nats")

# bin-count sweep on two datasets
B_LIST = [8, 16, 32, 64, 128, 256] if not QUICK else [16, 64]
ar_sweep = {}
for name in ["8gauss", "checkerboard"]:
    ar_sweep[name] = {"train": [], "test": []}
    for B in B_LIST:
        m, _ = cached(f"ar{B}_{name}", lambda B=B: AR2D(B),
                      lambda mdl, name=name: train_nn(mdl, lambda md, xb: md.nll_disc(xb).mean(), TRAIN[name], AR_STEPS))
        with torch.no_grad():
            ar_sweep[name]["train"].append(-m.log_prob(TRAIN[name]).mean().item())
            ar_sweep[name]["test"].append(-m.log_prob(TEST[name]).mean().item())
    print(name, "bins", B_LIST, "test NLL", [round(v, 3) for v in ar_sweep[name]["test"]])
RESULTS["ar"] = {"bins": 64, "steps": AR_STEPS, **ar_info,
                 "bin_sweep": {"B": B_LIST, **{n: {k: [rnd(v) for v in vals] for k, vals in d.items()} for n, d in ar_sweep.items()}}}
save_results()

In [ ]:
fig = plt.figure(figsize=(17, 4.8))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.5], wspace=0.28)
gen = torch.Generator(device=device).manual_seed(11)
m = AR_MODELS["checkerboard"]
with torch.no_grad():
    p1 = m.logit1.softmax(0).cpu().numpy()
    cond = m.net(m.emb(torch.arange(64, device=device))).softmax(1).cpu().numpy()   # rows: i1, cols: i2
    joint = p1[:, None] * cond
ax = fig.add_subplot(gs[0, 0])
ax.bar(np.linspace(-LIM, LIM, 64, endpoint=False) + 0.0625, p1, width=0.125, color="#2774B8")
ax.set_title(r"$p(x_1)$ (64 bins)"); ax.set_xlabel(r"$x_1$")
ax = fig.add_subplot(gs[0, 1])
ax.imshow(joint.T, origin="lower", extent=[-LIM, LIM, -LIM, LIM], cmap="viridis")
ax.set_title(r"$p(x_1)\,p(x_2\mid x_1)$"); ax.set_xticks([-4, 0, 4]); ax.set_yticks([-4, 0, 4])
ax = fig.add_subplot(gs[0, 2])
s = m.sample(5000, gen).cpu()
ax.scatter(s[:, 0], s[:, 1], s=1.2, alpha=0.5, color="#E97820")
ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_title("samples: $x_1$ then $x_2$")
ax.set_xticks([-4, 0, 4]); ax.set_yticks([-4, 0, 4])
ax = fig.add_subplot(gs[0, 3])
for c, name in zip(["#2774B8", "#3D8C54"], ["8gauss", "checkerboard"]):
    ax.plot(B_LIST, ar_sweep[name]["test"], "o-", color=c, lw=2.5, label=f"{TOY_TITLE[name]} test")
    ax.plot(B_LIST, ar_sweep[name]["train"], ":", color=c, lw=1.8)
    ax.axhline(RESULTS["toy"]["true_test_nll_nats"][name], color=c, ls="--", lw=1.2)
ax.set_xscale("log", base=2); ax.set_xlabel("bins per dimension B"); ax.set_ylabel("continuous NLL (nats)")
ax.set_title("quantization vs data per bin"); ax.legend(fontsize=12); ax.grid(alpha=0.3)
savefig(fig, "ar_model")

**관찰 포인트**: checkerboard 의 경계는 bin 경계와 맞기 때문에 AR 모델은 checkerboard 를 거의 완벽하게 표현합니다.
8 Gaussians 는 $\sigma=0.15$ 인데 bin 이 넓으면 ($B\le16$, $\Delta\ge0.5$) 봉우리를 뭉개 NLL 이 크고, $B$ 를 키우면 entropy 하한에 다가갑니다.
$B=256$ 에서는 bin 당 sample 이 적어 train (점선) 과 test 가 벌어지기 시작합니다. 같은 trade-off 가 이미지의 8-bit 와 **token 화** (VQ, L03) 에 있습니다.

## 5. Normalizing flow: change of variables 와 정확한 log-likelihood

가역 (bijective) 이고 미분 가능한 $f_\theta:x\mapsto z$ 와 단순한 base $p_Z=\mathcal N(0,I)$ 를 쓰면

$$
p_X(x)=p_Z\big(f_\theta(x)\big)\,\Big|\det\frac{\partial f_\theta(x)}{\partial x}\Big|
\quad\Longrightarrow\quad
\log p_X(x)=\log p_Z(f_\theta(x))+\sum_{l=1}^{L}\log\Big|\det\frac{\partial h_l}{\partial h_{l-1}}\Big|,\quad f=h_L\circ\cdots\circ h_1 .
$$

(1D 직관: $p_X(x)\,dx=p_Z(z)\,dz$ — 확률 질량 보존. 공간을 늘리면 density 는 줄어듦.) 핵심 제약은 **Jacobian determinant 를 싸게** 계산하는 것입니다.

**Affine coupling** (RealNVP, Dinh et al., ICLR 2017): 좌표를 $(x_a,x_b)$ 로 나누고

$$
y_a=x_a,\qquad y_b=x_b\odot\exp\big(s_\theta(x_a)\big)+t_\theta(x_a)
\qquad\Rightarrow\qquad
\frac{\partial y}{\partial x}=\begin{pmatrix}I&0\\ \ast&\mathrm{diag}(e^{s_\theta(x_a)})\end{pmatrix},\quad
\log|\det|=\sum_j s_\theta(x_a)_j .
$$

삼각 행렬이므로 determinant 는 대각의 곱 — $s_\theta,t_\theta$ 는 **아무 neural net** 이어도 됩니다 (역함수를 구할 필요가 없음).
역변환도 닫힌 형태 $x_b=(y_b-t_\theta(y_a))\odot e^{-s_\theta(y_a)}$ 라 sampling 은 한 번의 역방향 pass 입니다.
층마다 $a,b$ 역할을 바꾸고, 그 사이에 고정된 회전 (orthogonal, $\log|\det|=0$; Glow 의 1×1 conv 에 해당, Kingma & Dhariwal, NeurIPS 2018) 을 넣어 섞습니다.

한계: 모든 층이 가역이라 **차원을 줄일 수 없고** ($z$ 와 $x$ 가 같은 차원), 위상 (topology) 을 바꿀 수 없습니다 — 하나의 Gaussian blob 을 8 개로 찢으려면 얇은 "다리" 가 남습니다.
Continuous normalizing flow / flow matching (L08) 은 이 이산 층을 ODE $\dot x=v_\theta(x,t)$ 로 바꾼 것입니다.

In [ ]:
class Coupling(nn.Module):
    """Affine coupling in 2D: dimension `a` conditions dimension `b`."""

    def __init__(self, a, h=256):
        super().__init__()
        self.a, self.b = a, 1 - a
        self.net = nn.Sequential(nn.Linear(1, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, 2))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)   # start at the identity map

    def st(self, xa):
        s, t = self.net(xa).chunk(2, dim=1)
        return 2.0 * torch.tanh(s / 2.0), t            # soft-clamped log-scale for stability

    def forward(self, x):                              # x -> z direction, returns log|det|
        xa, xb = x[:, self.a:self.a + 1], x[:, self.b:self.b + 1]
        s, t = self.st(xa)
        yb = xb * torch.exp(s) + t
        y = torch.cat([xa, yb], 1) if self.a == 0 else torch.cat([yb, xa], 1)
        return y, s[:, 0]

    def inverse(self, y):
        ya, yb = y[:, self.a:self.a + 1], y[:, self.b:self.b + 1]
        s, t = self.st(ya)
        xb = (yb - t) * torch.exp(-s)
        return torch.cat([ya, xb], 1) if self.a == 0 else torch.cat([xb, ya], 1)


class RealNVP2D(nn.Module):
    def __init__(self, n_blocks=8, h=256, seed=0):
        super().__init__()
        self.layers = nn.ModuleList()
        gen = torch.Generator().manual_seed(seed)
        rots = []
        for i in range(n_blocks):
            self.layers.append(Coupling(0, h)); self.layers.append(Coupling(1, h))
            th = torch.rand(1, generator=gen).item() * math.pi
            rots.append(torch.tensor([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]]))
        self.register_buffer("rots", torch.stack(rots))   # fixed rotation after every pair, log|det| = 0

    def forward(self, x):                              # data -> latent, total log|det|
        ld = torch.zeros(len(x), device=x.device)
        for i in range(0, len(self.layers), 2):
            x, l1 = self.layers[i](x); x, l2 = self.layers[i + 1](x)
            x = x @ self.rots[i // 2].T
            ld = ld + l1 + l2
        return x, ld

    def log_prob(self, x):
        z, ld = self(x)
        return -0.5 * (z ** 2).sum(1) - math.log(2 * math.pi) + ld

    @torch.no_grad()
    def sample(self, n, gen, return_path=False):
        z = torch.randn(n, 2, device=device, generator=gen)
        path = [z]
        for i in reversed(range(0, len(self.layers), 2)):
            z = z @ self.rots[i // 2]                   # inverse rotation
            z = self.layers[i + 1].inverse(z); z = self.layers[i].inverse(z)
            path.append(z)
        return (z, path) if return_path else z


FLOW_STEPS = 12000 if not QUICK else 1000
FLOW_MODELS, flow_info = {}, {}
for name in TOYS:
    m, info = cached(f"realnvp_{name}", lambda: RealNVP2D(8, 256),
                     lambda mdl, name=name: train_nn(mdl, lambda md, xb: -md.log_prob(xb).mean(), TRAIN[name],
                                                      FLOW_STEPS, bs=1024, lr=5e-4, log_every=100))
    m.eval(); FLOW_MODELS[name] = m
    with torch.no_grad():
        flow_info[name] = {"test_nll": rnd(-m.log_prob(TEST[name]).mean()),
                           "train_nll": rnd(-m.log_prob(TRAIN[name][:20000]).mean()),
                           "train_s": rnd(info["train_s"], 1) if info else None}
    print(f"RealNVP {name}: test NLL {flow_info[name]['test_nll']:.3f} nats", "(trained)" if info else "(loaded)")
n_par = sum(p.numel() for p in FLOW_MODELS["8gauss"].parameters())
RESULTS["flow"] = {"blocks": 8, "couplings": 16, "hidden": 256, "params": n_par, "steps": FLOW_STEPS, **flow_info}

# Exactness check: inverse(forward(x)) == x, and log|det| equals autograd's Jacobian determinant
m = FLOW_MODELS["checkerboard"]
xx = TEST["checkerboard"][:256].clone()
with torch.no_grad():
    z, ld = m(xx)
    gen = torch.Generator(device=device).manual_seed(0)
    xr = z.clone()
    for i in reversed(range(0, len(m.layers), 2)):
        xr = xr @ m.rots[i // 2]; xr = m.layers[i + 1].inverse(xr); xr = m.layers[i].inverse(xr)
inv_err = (xr - xx).abs().max().item()
J = torch.autograd.functional.jacobian(lambda v: m(v[None])[0][0], xx[0])
ld_err = abs(math.log(abs(torch.det(J).item())) - ld[0].item())
RESULTS["flow"].update({"inverse_max_abs_err": inv_err, "logdet_vs_autograd_abs_err": ld_err})
print(f"flow inverse error {inv_err:.2e}, log|det| vs autograd {ld_err:.2e}")
save_results()

In [ ]:
# Figure: how the flow pushes N(0, I) through the layers (checkerboard)
gen = torch.Generator(device=device).manual_seed(5)
_, path = FLOW_MODELS["checkerboard"].sample(6000, gen, return_path=True)
show = [0, 2, 4, 6, 8]
fig, axs = plt.subplots(1, len(show), figsize=(17, 3.9))
col = path[0][:, 0].cpu()      # colour by the latent x-coordinate to follow points
for ax, j in zip(axs, show):
    p = path[j].cpu()
    ax.scatter(p[:, 0], p[:, 1], s=1.2, c=col, cmap="coolwarm", alpha=0.7)
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("z ~ N(0, I)" if j == 0 else f"after {2 * j} of 16 couplings")
savefig(fig, "flow_layers")

# GIF of the same push-forward
try:
    import imageio.v2 as imageio
    frames = []
    for j in range(len(path)):
        f, ax = plt.subplots(figsize=(4.2, 4.2))
        p = path[j].cpu()
        ax.scatter(p[:, 0], p[:, 1], s=1, c=col, cmap="coolwarm", alpha=0.7)
        ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"RealNVP inverse pass: block {j}/8")
        f.canvas.draw(); frames.append(np.asarray(f.canvas.buffer_rgba())[..., :3].copy()); plt.close(f)
    frames = frames + [frames[-1]] * 4
    imageio.mimsave(os.path.join(MEDIA, "L01_flow_layers.gif"), frames, duration=0.5, loop=0)
except Exception as e:  # imageio missing or GIF writer error should not stop the tutorial
    print("GIF skipped:", e)

**관찰 포인트**: 점의 색은 latent 의 $z_1$ 좌표입니다. Coupling 하나는 한 좌표만 바꾸지만, 회전과 번갈아 쌓으면 Gaussian 이 접히고 늘어나 8 개 정사각형이 됩니다.
정사각형 사이의 빈칸은 연속 bijection 으로 정확히 만들 수 없어서, density 그림 (§7) 에 얇은 "다리" 가 남습니다.
Inverse 오차 (`inverse_max_abs_err`) 와 log-det 대 autograd Jacobian 의 오차가 float 정밀도 수준인 것을 확인하세요 — **likelihood 가 근사가 아니라 정확** 합니다.

## 6. GAN: likelihood 없이 sample 만 — 그리고 mode collapse

GAN (Goodfellow et al., NeurIPS 2014) 은 $x=G_\theta(z),\ z\sim\mathcal N(0,I)$ 인 **implicit model**: sampling 은 쉽지만 $p_\theta(x)$ 를 계산할 수 없습니다
(차원을 바꾸는 $G$ 는 가역이 아님). 대신 판별자 $D$ 와 게임을 합니다:

$$
\min_G\max_D\;V(D,G)=\mathbb E_{x\sim p}\big[\log D(x)\big]+\mathbb E_{z}\big[\log(1-D(G(z)))\big].
$$

**최적 판별자**: 고정된 $G$ (생성 분포 $q$) 에서 점별로 $a\log D+b\log(1-D)$ 를 최대화하면 $D^\star(x)=\frac{p(x)}{p(x)+q(x)}$. 대입하면

$$
V(D^\star,G)=\mathbb E_p\Big[\log\frac{p}{p+q}\Big]+\mathbb E_q\Big[\log\frac{q}{p+q}\Big]
=-\log4+\mathrm{KL}\Big(p\,\Big\|\,\tfrac{p+q}2\Big)+\mathrm{KL}\Big(q\,\Big\|\,\tfrac{p+q}2\Big)=-\log 4+2\,\mathrm{JS}(p\|q).
$$

즉 이상적으로는 Jensen–Shannon divergence 를 줄입니다 (forward 와 reverse KL 의 대칭 혼합). 실제로는 $G$ 의 gradient 가 포화되지 않도록
**non-saturating** loss $\min_G -\mathbb E\log D(G(z))$ 를 쓰고 (본 코드), $D$ 가 최적이 아닌 상태에서 번갈아 SGD 합니다.

**Mode collapse**: $G$ 의 update 는 "지금의 $D$ 가 진짜라고 믿는 곳" 으로 sample 을 옮깁니다. $G$ 가 한 mode 로 몰리면 $D$ 가 그 mode 를 의심하게 되고, $G$ 는 다른 mode 로 이동 —
모든 mode 를 동시에 덮게 하는 항이 목적함수에 직접 없습니다 (forward KL 처럼 "데이터 점에서 density 가 0 이면 무한대 벌점" 이 없음).
8 Gaussians 에서 **같은 모델, 같은 loss** 로 learning rate 만 바꿔 비교합니다 (Adam, $\beta_1=0.5$, $G$:$D$ = 1:1 step, cosine lr decay):

- **lr = 1e-3** (빠른 update): $G$ 와 $D$ 가 서로를 쫓아 다니며 mode 사이를 오가는 **mode hopping** 과 mode 누락을 기대
- **lr = 2e-4** (DCGAN 권장값, Radford et al., ICLR 2016): 8 개 mode 를 고르게 덮기를 기대

(`train_gan(kind="wgan", lr=1e-4, n_critic=5)` 로 WGAN-GP (Gulrajani et al., NeurIPS 2017) 도 돌릴 수 있습니다 — Exercise 6.)

Mode 지표: 생성 sample 중 가장 가까운 중심과의 거리가 $3\sigma$ 이내면 "high quality", 각 mode 에 들어간 sample 이 기대치 ($N/8$) 의 10 % 이상이면 "covered".

In [ ]:
def mlp(i, o, h=256, L=3, act=nn.LeakyReLU(0.2)):
    layers, d = [], i
    for _ in range(L):
        layers += [nn.Linear(d, h), copy.deepcopy(act)]; d = h
    return nn.Sequential(*layers, nn.Linear(d, o))


def mode_stats(x):
    """(#modes covered, fraction high-quality, per-mode counts) for the 8-Gaussians task."""
    d = torch.cdist(x, GAUSS_C)
    m = d.min(1)
    hq = m.values < 3 * GAUSS_S
    counts = torch.bincount(m.indices[hq], minlength=8)
    return int((counts > 0.1 * len(x) / 8).sum()), hq.float().mean().item(), counts.tolist()


def train_gan(x_train, kind, steps, lr, n_critic, bs=512, zdim=2, seed=0, snap_every=0):
    torch.manual_seed(seed)
    G, D = mlp(zdim, 2).to(device), mlp(2, 1).to(device)
    betas = (0.5, 0.999) if kind == "ns" else (0.5, 0.9)
    oG = torch.optim.Adam(G.parameters(), lr, betas=betas)
    oD = torch.optim.Adam(D.parameters(), lr, betas=betas)
    gen = torch.Generator(device=device).manual_seed(seed)
    sG = torch.optim.lr_scheduler.CosineAnnealingLR(oG, steps)
    sD = torch.optim.lr_scheduler.CosineAnnealingLR(oD, steps)
    zfix = torch.randn(4000, zdim, device=device, generator=torch.Generator(device=device).manual_seed(99))
    snaps = []
    for it in range(steps + 1):
        if snap_every and it % snap_every == 0:
            with torch.no_grad():
                snaps.append((it, G(zfix).cpu()))
        if it == steps:
            break
        for _ in range(n_critic):
            xr = x_train[torch.randint(0, len(x_train), (bs,), device=device, generator=gen)]
            xf = G(torch.randn(bs, zdim, device=device, generator=gen)).detach()
            if kind == "wgan":
                a = torch.rand(bs, 1, device=device, generator=gen)
                xi = (a * xr + (1 - a) * xf).requires_grad_(True)
                gr = torch.autograd.grad(D(xi).sum(), xi, create_graph=True)[0]
                lD = D(xf).mean() - D(xr).mean() + 10.0 * ((gr.norm(dim=1) - 1) ** 2).mean()
            else:
                lD = F.softplus(-D(xr)).mean() + F.softplus(D(xf)).mean()   # -log D(x) - log(1 - D(G(z)))
            oD.zero_grad(set_to_none=True); lD.backward(); oD.step()
        xf = G(torch.randn(bs, zdim, device=device, generator=gen))
        lG = -D(xf).mean() if kind == "wgan" else F.softplus(-D(xf)).mean()   # non-saturating: -log D(G(z))
        oG.zero_grad(set_to_none=True); lG.backward(); oG.step()
        sG.step(); sD.step()
    return G, snaps


GAN_CFG = {"fast": dict(kind="ns", lr=1e-3, n_critic=1), "ns": dict(kind="ns", lr=2e-4, n_critic=1)}
GAN_LAB = {"fast": "NS-GAN, lr 1e-3", "ns": "NS-GAN, lr 2e-4"}
GAN_STEPS = 20000 if not QUICK else 2000
SNAP = GAN_STEPS // 40
gan_traj = {}
gan_res = {}
for cname, cfg in GAN_CFG.items():
    path = os.path.join(RUNS, f"gan_{cname}_8gauss.pt")
    if os.path.exists(path):
        ck = torch.load(path, map_location=device)
        G = mlp(2, 2).to(device); G.load_state_dict(ck["G"]); tr = None
        snaps = [(it, sn.cpu()) for it, sn in ck["snaps"]]
    else:
        t0 = time.time()
        G, snaps = train_gan(TRAIN["8gauss"], steps=GAN_STEPS, snap_every=SNAP, **cfg)
        tr = time.time() - t0
        torch.save({"G": G.state_dict(), "snaps": snaps}, path)
    gan_traj[cname] = snaps
    traj = [(it, *mode_stats(s.to(device))[:2]) for it, s in snaps]
    gan_res[cname] = {"steps": GAN_STEPS, "train_s": rnd(tr, 1) if tr else None,
                      "modes_covered_traj": [t[1] for t in traj], "hq_frac_traj": [rnd(t[2], 3) for t in traj],
                      "snap_iters": [t[0] for t in traj], "final_counts": mode_stats(snaps[-1][1].to(device))[2]}
    print(f"GAN {cname}: modes covered over training {gan_res[cname]['modes_covered_traj']}")
RESULTS["gan_8gauss"] = gan_res
save_results()

In [ ]:
fig = plt.figure(figsize=(17, 8.2))
gs = fig.add_gridspec(2, 6, width_ratios=[1, 1, 1, 1, 1, 1.6], wspace=0.22, hspace=0.3)
xs_cpu = TRAIN["8gauss"][:3000].cpu()
for r, cname in enumerate(["fast", "ns"]):
    lab = GAN_LAB[cname]
    snaps = gan_traj[cname]
    pick = [len(snaps) // 8, len(snaps) // 4, len(snaps) // 2, 3 * len(snaps) // 4, len(snaps) - 1]
    for j, k in enumerate(pick):
        ax = fig.add_subplot(gs[r, j])
        it, s = snaps[k]
        ax.scatter(xs_cpu[:, 0], xs_cpu[:, 1], s=1, color="#B4C3D6", alpha=0.6)
        ax.scatter(s[:, 0], s[:, 1], s=1.5, color="#E97820" if cname == "fast" else "#3D8C54", alpha=0.6)
        nm, hq, _ = mode_stats(s.to(device))
        ax.set_title(f"step {it // 1000}k: {nm}/8", fontsize=14)
        ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(lab, fontsize=14)
    ax = fig.add_subplot(gs[r, 5])
    ax.plot(gan_res[cname]["snap_iters"], gan_res[cname]["modes_covered_traj"], "-", lw=2.5,
            color="#E97820" if cname == "fast" else "#3D8C54", label="modes covered")
    ax.set_ylim(0, 8.5); ax.set_ylabel("modes covered (of 8)"); ax.set_xlabel("generator step")
    ax2 = ax.twinx()
    ax2.plot(gan_res[cname]["snap_iters"], gan_res[cname]["hq_frac_traj"], ":", lw=2, color="#404040")
    ax2.set_ylim(0, 1.05); ax2.set_ylabel("high-quality fraction (dotted)")
    ax.set_title(lab)
savefig(fig, "gan_collapse")

try:
    import imageio.v2 as imageio
    frames = []
    for (it, a), (_, b) in zip(gan_traj["fast"], gan_traj["ns"]):
        f, axs = plt.subplots(1, 2, figsize=(7.6, 4.0))
        for ax, s, c, lab in zip(axs, (a, b), ("#E97820", "#3D8C54"), (GAN_LAB["fast"], GAN_LAB["ns"])):
            ax.scatter(xs_cpu[:, 0], xs_cpu[:, 1], s=1, color="#B4C3D6", alpha=0.6)
            ax.scatter(s[:, 0], s[:, 1], s=1, color=c, alpha=0.6)
            ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"{lab}: step {it}", fontsize=11)
        f.canvas.draw(); frames.append(np.asarray(f.canvas.buffer_rgba())[..., :3].copy()); plt.close(f)
    imageio.mimsave(os.path.join(MEDIA, "L01_gan_training.gif"), frames, duration=0.25, loop=0)
except Exception as e:
    print("GIF skipped:", e)

**관찰 포인트**: 위 행 (lr 1e-3) 의 snapshot 들을 비교하세요 — 일부 mode 만 덮거나 덮는 mode 가 시간에 따라 바뀌면 mode collapse / mode hopping 입니다.
"High-quality fraction" (점선) 은 높게 유지되는데 mode 수는 낮을 수 있습니다: **sample 하나하나는 그럴듯하지만 다양성이 없는** 상태 — 이것이 §7 의 precision 대 recall 입니다.
GAN 은 test NLL 을 계산할 수 없으므로, 이런 실패는 sample 기반 지표로만 잡을 수 있습니다.

### 6.1 네 데이터 모두에 GAN 학습

비교 표를 위해 lr 2e-4 설정으로 네 데이터를 모두 학습합니다 (8 Gaussians 는 위에서 학습한 모델 재사용).

In [ ]:
GAN_MODELS = {}
for name in TOYS:
    if name == "8gauss":
        G = mlp(2, 2).to(device)
        G.load_state_dict(torch.load(os.path.join(RUNS, "gan_ns_8gauss.pt"), map_location=device)["G"])
    else:
        path = os.path.join(RUNS, f"gan_ns_{name}.pt")
        if os.path.exists(path):
            G = mlp(2, 2).to(device); G.load_state_dict(torch.load(path, map_location=device)["G"])
        else:
            G, _ = train_gan(TRAIN[name], steps=GAN_STEPS, **GAN_CFG["ns"])
            torch.save({"G": G.state_dict(), "snaps": []}, path)
    G.eval(); GAN_MODELS[name] = G

## 7. 모델 비교: NLL 과 sample 기반 지표

**NLL / bits-per-dim**: likelihood 모델 (GMM, AR, flow) 은 test NLL 로 직접 비교합니다 (같은 데이터, 같은 단위 = nats / sample, $D=2$).
연속 데이터의 NLL 은 음수일 수도 있습니다 (density > 1). 이미지에서는 $D$ 로 나누고 $\ln2$ 로 나눠 **bits/dim** 으로 보고합니다.

**MMD** (Gretton et al., JMLR 2012): kernel $k$ 의 feature 공간에서 두 분포의 평균 embedding 사이 거리

$$
\mathrm{MMD}^2(p,q)=\mathbb E_{x,x'\sim p}k(x,x')+\mathbb E_{y,y'\sim q}k(y,y')-2\,\mathbb E_{x\sim p,y\sim q}k(x,y),
$$

characteristic kernel (RBF) 이면 $\mathrm{MMD}=0\iff p=q$. 같은 분포끼리의 항에서 $i=j$ 를 빼면 **unbiased** 추정량이 됩니다 (그래서 음수도 가능).
여기서는 RBF bandwidth $\{0.1,0.2,0.5,1\}$ 의 합을 씁니다.

**Precision / recall** (Kynkäänniemi et al., NeurIPS 2019): 실제 sample 각각에 $k$-번째 최근접 이웃 거리를 반지름으로 하는 공을 놓아 "real manifold" 를 근사합니다.

$$
\text{precision}=\frac1M\sum_{j}\mathbb 1\big[\hat x_j\in\textstyle\bigcup_i B(x_i,r_k(x_i))\big],\qquad
\text{recall}=\frac1N\sum_{i}\mathbb 1\big[x_i\in\textstyle\bigcup_j B(\hat x_j,r_k(\hat x_j))\big].
$$

Precision = "생성 sample 이 그럴듯한가" (quality), recall = "실제 데이터의 다양성을 덮는가" (coverage). $k=3$ 사용.
먼저 **정답을 아는 조작** 으로 지표가 의도대로 반응하는지 확인합니다: 8 Gaussians 에서 (a) mode 를 일부 버린 분포, (b) 각 mode 의 $\sigma$ 를 키운 분포.

In [ ]:
def mmd2_rbf(X, Y, bws=(0.1, 0.2, 0.5, 1.0)):
    """Unbiased MMD^2 with a sum of RBF kernels."""
    def k(A, B):
        d2 = torch.cdist(A, B) ** 2
        return sum(torch.exp(-d2 / (2 * b * b)) for b in bws)
    n, m = len(X), len(Y)
    kxx, kyy, kxy = k(X, X), k(Y, Y), k(X, Y)
    return ((kxx.sum() - kxx.diagonal().sum()) / (n * (n - 1)) + (kyy.sum() - kyy.diagonal().sum()) / (m * (m - 1))
            - 2 * kxy.mean()).item()


def knn_radius(X, k, chunk=2048):
    r = []
    for i in range(0, len(X), chunk):
        d = torch.cdist(X[i:i + chunk], X)
        r.append(d.kthvalue(k + 1, dim=1).values)          # +1: the point itself is at distance 0
    return torch.cat(r)


def in_manifold(Q, X, rX, chunk=2048):
    out = []
    for i in range(0, len(Q), chunk):
        d = torch.cdist(Q[i:i + chunk], X)
        out.append((d <= rX[None]).any(1))
    return torch.cat(out)


def precision_recall(real, fake, k=3):
    real, fake = real.float(), fake.float()
    r_real, r_fake = knn_radius(real, k), knn_radius(fake, k)
    return in_manifold(fake, real, r_real).float().mean().item(), in_manifold(real, fake, r_fake).float().mean().item()


N_EVAL = 10000 if not QUICK else 3000
real_eval = TEST["8gauss"][:N_EVAL]
sanity = {"drop_modes": [], "widen": []}
g = torch.Generator(device=device).manual_seed(21)
for m_keep in [8, 6, 4, 2, 1]:
    k = torch.randint(0, m_keep, (N_EVAL,), device=device, generator=g)
    fake = GAUSS_C[k] + GAUSS_S * torch.randn(N_EVAL, 2, device=device, generator=g)
    p, r = precision_recall(real_eval, fake)
    sanity["drop_modes"].append({"modes": m_keep, "precision": rnd(p, 3), "recall": rnd(r, 3), "mmd2": rnd(mmd2_rbf(real_eval[:5000], fake[:5000]), 5)})
for fac in [1.0, 1.5, 2.0, 3.0, 4.0]:
    k = torch.randint(0, 8, (N_EVAL,), device=device, generator=g)
    fake = GAUSS_C[k] + fac * GAUSS_S * torch.randn(N_EVAL, 2, device=device, generator=g)
    p, r = precision_recall(real_eval, fake)
    sanity["widen"].append({"sigma_factor": fac, "precision": rnd(p, 3), "recall": rnd(r, 3), "mmd2": rnd(mmd2_rbf(real_eval[:5000], fake[:5000]), 5)})
RESULTS["metric_sanity_8gauss"] = sanity
print(json.dumps(sanity, indent=1))
save_results()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
dm = sanity["drop_modes"]; wd = sanity["widen"]
axs[0].plot([d["modes"] for d in dm], [d["precision"] for d in dm], "o-", lw=2.5, color="#2774B8", label="precision")
axs[0].plot([d["modes"] for d in dm], [d["recall"] for d in dm], "s-", lw=2.5, color="#E97820", label="recall")
axs[0].plot([d["modes"] for d in dm], [d["modes"] / 8 for d in dm], ":", color="#404040", label="modes kept / 8")
axs[0].set_xlabel("modes kept by the 'model' (of 8)"); axs[0].set_title("(a) mode dropping"); axs[0].invert_xaxis()
axs[1].plot([d["sigma_factor"] for d in wd], [d["precision"] for d in wd], "o-", lw=2.5, color="#2774B8", label="precision")
axs[1].plot([d["sigma_factor"] for d in wd], [d["recall"] for d in wd], "s-", lw=2.5, color="#E97820", label="recall")
axs[1].set_xlabel(r"$\sigma_{\mathrm{model}}/\sigma_{\mathrm{data}}$"); axs[1].set_title("(b) blurry / too-wide modes")
for ax in axs:
    ax.set_ylim(0, 1.05); ax.grid(alpha=0.3); ax.legend(loc="lower left")
axs2 = axs[1].twinx()
axs2.plot([d["sigma_factor"] for d in wd], [d["mmd2"] for d in wd], "^--", color="#3D8C54", lw=1.8)
axs2.set_ylabel(r"MMD$^2$ (green, dashed)", color="#3D8C54")
axs3 = axs[0].twinx()
axs3.plot([d["modes"] for d in dm], [d["mmd2"] for d in dm], "^--", color="#3D8C54", lw=1.8)
axs3.set_ylabel(r"MMD$^2$ (green, dashed)", color="#3D8C54")
fig.tight_layout()
savefig(fig, "pr_sanity")

**관찰 포인트**: mode 를 버리면 precision 은 ≈ 1 로 유지되고 recall 만 떨어집니다 (GAN 의 mode collapse 형태). $\sigma$ 를 키우면 반대로 recall 은 유지되고 precision 이 떨어집니다
(forward-KL 모델의 "흐릿함" 형태). MMD 는 두 실패 모두에서 커지지만 **어느 쪽인지** 는 알려 주지 않습니다 — 그래서 하나의 숫자 (FID 포함) 대신 두 축을 함께 보고합니다.

In [ ]:
# Full comparison: every model x every dataset
GMM_LAB = f"GMM (EM, K={K_LIST[-1]})"
MODELS = {GMM_LAB: ("gmm", GMM_BEST), "AR (64 bins)": ("ar", AR_MODELS),
          "RealNVP": ("flow", FLOW_MODELS), "GAN": ("gan", GAN_MODELS)}


def model_sample(kind, mdl, n, gen):
    with torch.no_grad():
        if kind == "gmm":
            return gmm_sample(mdl, n, gen)
        if kind in ("ar", "flow"):
            return mdl.sample(n, gen)
        return mdl(torch.randn(n, 2, device=device, generator=gen))


def model_logp(kind, mdl, x):
    with torch.no_grad():
        if kind == "gmm":
            return gmm_logp(mdl, x)
        if kind in ("ar", "flow"):
            return mdl.log_prob(x)
    return None


table = {}
SAMPLES = {}
for mname, (kind, dct) in MODELS.items():
    table[mname] = {}
    for name in TOYS:
        gen = torch.Generator(device=device).manual_seed(123)
        s = model_sample(kind, dct[name], N_EVAL, gen)
        SAMPLES[(mname, name)] = s
        lp = model_logp(kind, dct[name], TEST[name])
        p, r = precision_recall(TEST[name][:N_EVAL], s)
        table[mname][name] = {"test_nll": rnd(-lp.mean()) if lp is not None else None,
                              "precision": rnd(p, 3), "recall": rnd(r, 3),
                              "mmd2": rnd(mmd2_rbf(TEST[name][:5000], s[:5000]), 5)}
# reference row: a second independent draw from the data distribution (sets the "perfect model" level)
table["data (2nd draw)"] = {}
for name in TOYS:
    s = sample_toy(name, N_EVAL, torch.Generator(device=device).manual_seed(777))
    SAMPLES[("data (2nd draw)", name)] = s
    lp = true_logp(name, TEST[name])
    p, r = precision_recall(TEST[name][:N_EVAL], s)
    table["data (2nd draw)"][name] = {"test_nll": rnd(-lp.mean()) if lp is not None else None, "precision": rnd(p, 3),
                                      "recall": rnd(r, 3), "mmd2": rnd(mmd2_rbf(TEST[name][:5000], s[:5000]), 5)}
RESULTS["toy_table"] = table
save_results()
for mname in table:
    print(f"{mname:18s}", " | ".join(f"{n}: nll={v['test_nll']} P={v['precision']} R={v['recall']} mmd={v['mmd2']}"
                                      for n, v in table[mname].items()))

In [ ]:
# Hero figure 1: samples of every model on every dataset
cols_order = ["data (2nd draw)", GMM_LAB, "AR (64 bins)", "RealNVP", "GAN"]
fig, axs = plt.subplots(4, 5, figsize=(16, 13.2))
for i, name in enumerate(TOYS):
    for j, mname in enumerate(cols_order):
        ax = axs[i, j]
        s = SAMPLES[(mname, name)][:5000].cpu()
        ax.scatter(s[:, 0], s[:, 1], s=0.8, alpha=0.5, color="#203864" if j == 0 else "#E97820")
        ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        v = table[mname][name]
        ax.text(0.02, 0.02, f"P {v['precision']:.2f}  R {v['recall']:.2f}", transform=ax.transAxes, fontsize=12,
                bbox=dict(facecolor="white", alpha=0.85, lw=0))
        if i == 0:
            ax.set_title(mname if j else "data", fontsize=16)
        if j == 0:
            ax.set_ylabel(TOY_TITLE[name], fontsize=16)
fig.suptitle("5,000 samples per model (seed 123); P / R = precision / recall vs 10k test points (k = 3)", y=0.995, fontsize=15)
fig.tight_layout()
savefig(fig, "samples_grid")

# Hero figure 2: densities of the three likelihood models (+ truth where known)
dens_models = [(GMM_LAB, "gmm"), ("AR (64 bins)", "ar"), ("RealNVP", "flow")]
fig, axs = plt.subplots(4, 4, figsize=(15, 15))
for i, name in enumerate(TOYS):
    tl = (lambda x, name=name: true_logp(name, x)) if true_logp(name, TEST[name][:2]) is not None else None
    panels = [("truth", tl)] + [(mn, (lambda x, mn=mn, kd=kd, name=name: model_logp(kd, MODELS[mn][1][name], x)))
                                       for mn, kd in dens_models]
    for j, (lab, fn) in enumerate(panels):
        ax = axs[i, j]
        if fn is None:
            s = TRAIN[name].cpu().numpy()
            H, _, _ = np.histogram2d(s[:, 0], s[:, 1], bins=150, range=[[-LIM, LIM], [-LIM, LIM]], density=True)
            Dn = H.T; lab = "train histogram"
            ax.text(0.03, 0.90, "histogram (no closed form)", transform=ax.transAxes, color="white", fontsize=12)
        else:
            Dn = density_grid(fn, n=300)
        ax.imshow(Dn, extent=[-LIM, LIM, -LIM, LIM], origin="lower", cmap="magma",
                  vmin=0, vmax=np.quantile(Dn, 0.995) + 1e-12)
        ax.set_xticks([]); ax.set_yticks([])
        nll = table[lab]["%s" % name]["test_nll"] if lab in table else (table["data (2nd draw)"][name]["test_nll"] if fn is not None else None)
        ttl = lab if i == 0 else ""
        if nll is not None:
            ax.text(0.03, 0.03, f"NLL {nll:.2f}", transform=ax.transAxes, color="white", fontsize=14,
                    bbox=dict(facecolor="black", alpha=0.7, lw=0))
        if ttl:
            ax.set_title(ttl, fontsize=16)
        if j == 0:
            ax.set_ylabel(TOY_TITLE[name], fontsize=16)
fig.suptitle("Learned densities p(x) (linear colour scale) and test NLL in nats / sample", y=0.995, fontsize=15)
fig.tight_layout()
savefig(fig, "densities_grid")

**관찰 포인트**

- **NLL 순위와 눈으로 본 품질이 항상 같지는 않습니다.** Density 그림에서 GMM 은 checkerboard 의 모서리를 둥글게, RealNVP 는 정사각형 사이에 얇은 다리를 남깁니다.
  AR (64 bins) 은 bin 해상도의 계단이 보이지만 checkerboard 에서는 경계가 bin 과 맞아 가장 좋습니다.
- GAN 은 NLL 칸이 비어 있습니다 (implicit model). P/R 로만 평가할 수 있습니다.
- "data (2nd draw)" 행은 **완벽한 모델의 점수** 입니다: precision/recall 은 1 이 아니라 k-NN 공 근사 때문에 0.9 대, MMD² 는 0 근처의 잡음입니다.
  지표의 절대값보다 이 기준선과의 차이를 읽어야 합니다.

## 8. FID on CIFAR-10: 정의, 측정, 민감도

고차원 이미지에서는 likelihood 를 안 주는 모델 (GAN, diffusion 의 sampler 대부분) 도 비교해야 해서 **sample 기반 거리** 를 씁니다.
**FID** (Heusel et al., NeurIPS 2017): ImageNet 으로 학습한 Inception-v3 의 pool3 feature (2048-d) 를 실제/생성 이미지에서 뽑아 각각 Gaussian $\mathcal N(\mu_r,\Sigma_r)$, $\mathcal N(\mu_g,\Sigma_g)$ 로 맞추고
두 Gaussian 사이의 Fréchet (= Wasserstein-2) 거리를 잽니다:

$$
\mathrm{FID}=\lVert\mu_r-\mu_g\rVert_2^2+\mathrm{Tr}\Big(\Sigma_r+\Sigma_g-2\big(\Sigma_r^{1/2}\Sigma_g\Sigma_r^{1/2}\big)^{1/2}\Big).
$$

(유도 스케치: 두 Gaussian 사이 $W_2^2=\min_{\text{coupling}}\mathbb E\|X-Y\|^2=\|\mu_r-\mu_g\|^2+\mathrm{Tr}\Sigma_r+\mathrm{Tr}\Sigma_g-2\max\mathrm{Tr}\,\mathrm{Cov}(X,Y)$,
마지막 항의 최댓값이 $\mathrm{Tr}(\Sigma_r^{1/2}\Sigma_g\Sigma_r^{1/2})^{1/2}$. $\mathrm{Tr}(\Sigma_r\Sigma_g)^{1/2}$ 와 같은 값이라 구현은 $\Sigma_r\Sigma_g$ 의 고윳값 제곱근 합을 씁니다.)

주의할 성질:

1. **Biased in $N$**: $\Sigma_g$ 를 유한 sample 로 추정하면 trace 항이 위로 치우칩니다 — 같은 분포여도 $N$ 이 작을수록 FID 가 큽니다. **sample 수가 다른 FID 는 비교 불가.**
2. Feature 가 ImageNet 분류기 것이므로 "사람 눈" 이 아니라 **Inception 이 보는 차이** 를 잽니다 (작은 blur/noise 에 매우 민감).
3. Precision/recall 처럼 quality 와 diversity 를 분리하지 않습니다.

**KID** (Bińkowski et al., ICLR 2018) 는 같은 feature 에 다항 kernel $k(x,y)=(x^\top y/d+1)^3$ 의 unbiased MMD² — $N$ 에 대한 bias 가 없습니다.

측정 계획 (reference = CIFAR-10 train 50k, torchmetrics `FrechetInceptionDistance(feature=2048)` 의 Inception 으로 feature 추출, FID 식은 직접 구현하고 torchmetrics 와 대조):

- real vs real: test 10k vs train 50k, 그리고 test 의 $N$ = 500 … 10k subset (5 번 반복) — 이상적인 생성 모델이 받을 점수와 $N$-bias
- Gaussian blur $\sigma\in\{0.5,1,1.5,2\}$ px, 가우시안 잡음 $\sigma\in\{2,5,10,20\}/255$ — 사람 눈에는 작은 변화가 FID 를 얼마나 바꾸는가
- 좌우 반전 (의미 보존) 과 **5 개 class 만** (mode dropping) — 같은 5k 장으로 비교

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
import torchvision.transforms.functional as TF

cif = np.load(CIFAR)
train_u8 = torch.from_numpy(cif["train_x"]).permute(0, 3, 1, 2).contiguous()     # (50000, 3, 32, 32) uint8
test_u8 = torch.from_numpy(cif["test_x"]).permute(0, 3, 1, 2).contiguous()
test_y = torch.from_numpy(cif["test_y"]).long()
classes = [str(c) for c in cif["classes"]]
if QUICK:
    train_u8, test_u8, test_y = train_u8[:10000], test_u8[:2000], test_y[:2000]
print("CIFAR-10:", tuple(train_u8.shape), tuple(test_u8.shape))

try:
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)
except Exception as e:
    raise SystemExit("[L01] could not build the FID Inception network. It needs "
                     "~/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth (torch-fidelity). " + str(e))
inception = fid_metric.inception


@torch.no_grad()
def incep_feats(u8, bs=200):            # small chunks keep the peak memory low (resized to 299x299 inside)
    out = []
    for i in range(0, len(u8), bs):
        out.append(inception(u8[i:i + bs].to(device)).double().cpu())
    return torch.cat(out)


def stats(f):
    mu = f.mean(0)
    fc = f - mu
    return mu, fc.T @ fc / (len(f) - 1)


def fid_from_stats(m1, s1, m2, s2):
    """Frechet distance; Tr((S1 S2)^{1/2}) = sum of sqrt of the eigenvalues of S1 S2 (real, >= 0 up to noise)."""
    ev = torch.linalg.eigvals(s1.to(device) @ s2.to(device))
    tr_sqrt = ev.sqrt().real.sum().item()
    return ((m1 - m2) ** 2).sum().item() + torch.trace(s1).item() + torch.trace(s2).item() - 2 * tr_sqrt


def kid(f1, f2, n_sub=1000, n_rep=50, seed=0):
    """Unbiased polynomial-kernel MMD^2 (KID) averaged over random subsets, x 1000 as usually reported."""
    gen = torch.Generator().manual_seed(seed)
    d = f1.shape[1]
    vals = []
    for _ in range(n_rep):
        a = f1[torch.randperm(len(f1), generator=gen)[:n_sub]].to(device)
        b = f2[torch.randperm(len(f2), generator=gen)[:n_sub]].to(device)
        kxx = (a @ a.T / d + 1) ** 3; kyy = (b @ b.T / d + 1) ** 3; kxy = (a @ b.T / d + 1) ** 3
        m = n_sub
        vals.append(((kxx.sum() - kxx.diagonal().sum()) / (m * (m - 1)) + (kyy.sum() - kyy.diagonal().sum()) / (m * (m - 1))
                     - 2 * kxy.mean()).item())
    return 1000 * float(np.mean(vals)), 1000 * float(np.std(vals))


feat_path = os.path.join(RUNS, "inception_train_test.pt")
t0 = time.time()
if os.path.exists(feat_path):
    ck = torch.load(feat_path)
    f_train, f_test = ck["train"], ck["test"]
else:
    f_train, f_test = incep_feats(train_u8), incep_feats(test_u8)
    torch.save({"train": f_train, "test": f_test}, feat_path)
t_feat = time.time() - t0
print(f"Inception features: train {tuple(f_train.shape)}, test {tuple(f_test.shape)} in {t_feat:.1f}s")
mu_tr, S_tr = stats(f_train)
fid_res = {"n_train_ref": len(f_train), "n_test": len(f_test), "feature_s": rnd(t_feat, 1)}
fid_res["test_vs_train"] = rnd(fid_from_stats(*stats(f_test), mu_tr, S_tr), 3)
fid_res["kid_test_vs_train_x1e3"] = [rnd(v, 3) for v in kid(f_test, f_train)]
half = len(f_train) // 2
perm = torch.randperm(len(f_train), generator=torch.Generator().manual_seed(0))
fid_res["train_half_vs_half"] = rnd(fid_from_stats(*stats(f_train[perm[:half]]), *stats(f_train[perm[half:]])), 3)
print("FID test vs train:", fid_res["test_vs_train"], "| train half vs half:", fid_res["train_half_vs_half"],
      "| KID x1e3:", fid_res["kid_test_vs_train_x1e3"])

# Cross-check with the torchmetrics implementation on the same images (test 10k vs a 10k train subset)
n_cc = min(10000, len(test_u8))
sub = perm[:n_cc]
cc_path = os.path.join(RUNS, "fid_crosscheck_torchmetrics.json")
if os.path.exists(cc_path):
    fid_tm = json.load(open(cc_path))["fid"]
else:
    fid_metric.reset()
    for i in range(0, n_cc, 200):
        fid_metric.update(train_u8[sub[i:i + 200]].to(device), real=True)
        fid_metric.update(test_u8[i:i + 200].to(device), real=False)
    fid_tm = fid_metric.compute().item()
    json.dump({"fid": fid_tm}, open(cc_path, "w"))
fid_ours = fid_from_stats(*stats(f_train[sub]), *stats(f_test[:n_cc]))
fid_res["crosscheck_10k_vs_10k"] = {"torchmetrics": rnd(fid_tm, 4), "ours": rnd(fid_ours, 4)}
print(f"cross-check (10k vs 10k): torchmetrics {fid_tm:.4f}  ours {fid_ours:.4f}")
del fid_metric
save_results()

In [ ]:
# (1) N-bias of FID vs KID: random test subsets of size N vs the full train reference
N_LIST = [500, 1000, 2000, 5000, 10000] if not QUICK else [500, 1000, 2000]
fid_n, kid_n = [], []
for N in N_LIST:
    fs, ks = [], []
    for rep in range(5):
        idx = torch.randperm(len(f_test), generator=torch.Generator().manual_seed(rep))[:N]
        fs.append(fid_from_stats(*stats(f_test[idx]), mu_tr, S_tr))
        ks.append(kid(f_test[idx], f_train, n_sub=min(1000, N), n_rep=20, seed=rep)[0])
    fid_n.append((float(np.mean(fs)), float(np.std(fs)))); kid_n.append((float(np.mean(ks)), float(np.std(ks))))
    print(f"N={N}: FID {np.mean(fs):.2f} ± {np.std(fs):.2f}   KIDx1e3 {np.mean(ks):.3f}")
fid_res["n_sweep"] = {"N": N_LIST, "fid_mean": [rnd(a, 3) for a, _ in fid_n], "fid_std": [rnd(b, 3) for _, b in fid_n],
                      "kid_mean_x1e3": [rnd(a, 4) for a, _ in kid_n], "kid_std_x1e3": [rnd(b, 4) for _, b in kid_n]}


# (2) corruptions of the 10k test images
def blur(u8, s):
    k = 2 * math.ceil(3 * s) + 1
    return TF.gaussian_blur(u8.float(), [k, k], [s, s]).round().clamp(0, 255).to(torch.uint8)


def noise(u8, s, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return (u8.float() + s * torch.randn(u8.shape, generator=gen)).round().clamp(0, 255).to(torch.uint8)


CORR = {"blur": [0.5, 1.0, 1.5, 2.0], "noise": [2, 5, 10, 20]}
corr_res = {"blur": [], "noise": []}
CORR_IMGS = {}
for kind, levels in CORR.items():
    for lv in levels:
        u8 = blur(test_u8, lv) if kind == "blur" else noise(test_u8, lv)
        CORR_IMGS[(kind, lv)] = u8[:8]
        cpath = os.path.join(RUNS, f"inception_{kind}_{lv}.pt")
        if os.path.exists(cpath):
            f = torch.load(cpath)
        else:
            f = incep_feats(u8); torch.save(f, cpath)
        pr = precision_recall(f_train[perm[:len(f)]].float().to(device), f.float().to(device), k=3)
        corr_res[kind].append({"level": lv, "fid": rnd(fid_from_stats(*stats(f), mu_tr, S_tr), 3),
                               "kid_x1e3": rnd(kid(f, f_train, n_rep=20)[0], 3),
                               "precision": rnd(pr[0], 3), "recall": rnd(pr[1], 3)})
        print(kind, lv, corr_res[kind][-1])
pr0 = precision_recall(f_train[perm[:len(f_test)]].float().to(device), f_test.float().to(device), k=3)
fid_res["test_precision_recall_k3"] = [rnd(pr0[0], 3), rnd(pr0[1], 3)]
fid_res["corruptions"] = corr_res

# (3) semantics-preserving flip vs mode dropping (5 of 10 classes), both with N = 5000
n5 = min(5000, len(f_test) // 2)
keep = torch.isin(test_y, torch.arange(5))
f_5cls = f_test[keep][:n5]
idx5 = torch.randperm(len(f_test), generator=torch.Generator().manual_seed(1))[:n5]
fpath = os.path.join(RUNS, "inception_hflip5k.pt")
if os.path.exists(fpath):
    f_flip5 = torch.load(fpath)
else:
    f_flip5 = incep_feats(test_u8[idx5].flip(-1)); torch.save(f_flip5, fpath)
fid_res["same_N_5000"] = {
    "random_5000": rnd(fid_from_stats(*stats(f_test[idx5]), mu_tr, S_tr), 3),
    "hflip_5000": rnd(fid_from_stats(*stats(f_flip5), mu_tr, S_tr), 3),
    "classes_0to4_5000": rnd(fid_from_stats(*stats(f_5cls), mu_tr, S_tr), 3),
    "classes_kept": classes[:5]}
pr5 = precision_recall(f_train[perm[:n5]].float().to(device), f_5cls.float().to(device), k=3)
pr5r = precision_recall(f_train[perm[:n5]].float().to(device), f_test[idx5].float().to(device), k=3)
fid_res["same_N_5000"].update({"pr_random": [rnd(v, 3) for v in pr5r], "pr_classes_0to4": [rnd(v, 3) for v in pr5]})
print("same N=5000:", fid_res["same_N_5000"])
RESULTS["fid_cifar10"] = fid_res
save_results()

In [ ]:
# (4) bits/dim of trivial likelihood models on CIFAR-10 test (8-bit pixels, discrete NLL)
D = 3 * 32 * 32
tr = train_u8.long(); te = test_u8.long()
bpd = {"uniform": 8.0}
h_global = torch.bincount(tr.flatten(), minlength=256).double() + 1.0          # one histogram for all subpixels
lp = torch.log(h_global / h_global.sum())
bpd["global_histogram"] = rnd(-lp[te.flatten()].mean().item() / math.log(2), 4)
cnt = torch.zeros(D, 256, dtype=torch.float64)                                  # one histogram per subpixel position
cnt.scatter_add_(1, tr.reshape(len(tr), D).T.contiguous(), torch.ones(D, len(tr), dtype=torch.float64))
lp_pos = torch.log((cnt + 1.0) / (cnt + 1.0).sum(1, keepdim=True))
bpd["per_position_histogram"] = rnd(-lp_pos.gather(1, te.reshape(len(te), D).T.contiguous()).mean().item() / math.log(2), 4)
RESULTS["cifar_bpd_baselines"] = bpd
print("CIFAR-10 test bits/dim of independent-pixel baselines:", bpd)
save_results()

In [ ]:
# Figures for §8
fig, axs = plt.subplots(1, 3, figsize=(18, 5.2))
ns = fid_res["n_sweep"]
axs[0].errorbar(ns["N"], ns["fid_mean"], yerr=ns["fid_std"], fmt="o-", lw=2.5, color="#2774B8", capsize=4, label="FID")
axs[0].set_xscale("log"); axs[0].set_xlabel("# test images N (vs 50k train)"); axs[0].set_ylabel("FID (real vs real)", color="#2774B8")
ax2 = axs[0].twinx()
ax2.errorbar(ns["N"], ns["kid_mean_x1e3"], yerr=ns["kid_std_x1e3"], fmt="s--", lw=2, color="#E97820", capsize=4)
ax2.set_ylabel("KID × 1000", color="#E97820"); ax2.axhline(0, color="#E97820", lw=0.8, ls=":")
axs[0].set_title("(a) same distribution, different N")
for ax, kind, xl in [(axs[1], "blur", "Gaussian blur σ (pixels)"), (axs[2], "noise", "noise σ (of 255)")]:
    lv = [0] + [d["level"] for d in corr_res[kind]]
    fv = [fid_res["test_vs_train"]] + [d["fid"] for d in corr_res[kind]]
    ax.plot(lv, fv, "o-", lw=2.5, color="#2774B8")
    for a, b in zip(lv, fv):
        ax.annotate(f"{b:.1f}", (a, b), textcoords="offset points", xytext=(-8, 8), fontsize=12)
    ax.set_xlabel(xl); ax.set_ylabel("FID vs train 50k"); ax.set_title(f"({'b' if kind == 'blur' else 'c'}) {kind}, test 10k")
    axp = ax.twinx()
    axp.plot(lv, [fid_res["test_precision_recall_k3"][0]] + [d["precision"] for d in corr_res[kind]], "^:", color="#3D8C54", lw=2, label="precision")
    axp.plot(lv, [fid_res["test_precision_recall_k3"][1]] + [d["recall"] for d in corr_res[kind]], "v:", color="#6D237C", lw=2, label="recall")
    axp.set_ylim(0, 1.05); axp.legend(loc="center right", fontsize=12)
for ax in axs:
    ax.grid(alpha=0.3)
fig.tight_layout()
savefig(fig, "fid_sensitivity")

# image strip: what the corruptions look like
cols = [("clean", None)] + [(f"blur σ={lv}", ("blur", lv)) for lv in CORR["blur"]] + [(f"noise σ={lv}", ("noise", lv)) for lv in CORR["noise"]]
fig, axs = plt.subplots(3, len(cols), figsize=(18, 6.4))
for j, (lab, key) in enumerate(cols):
    imgs = test_u8[:3] if key is None else CORR_IMGS[key][:3]
    for i in range(3):
        axs[i, j].imshow(imgs[i].permute(1, 2, 0).numpy(), interpolation="nearest")
        axs[i, j].axis("off")
    if key is None:
        fv = fid_res["test_vs_train"]
    else:
        fv = [d["fid"] for d in corr_res[key[0]] if d["level"] == key[1]][0]
    axs[0, j].set_title(f"{lab}\nFID {fv:.1f}", fontsize=13)
fig.suptitle("CIFAR-10 test images (32×32, shown enlarged) and FID of the whole corrupted 10k set vs train 50k", y=1.0)
fig.tight_layout()
savefig(fig, "fid_corruptions")

**관찰 포인트**

- Real-vs-real FID 는 0 이 아닙니다 (test 10k vs train 50k). 이것이 **완벽한 생성 모델의 FID 하한** 이며, 작은 $N$ 에서 FID 가 크게 부풀고 KID 는 0 근처에 머무는 것을 (a) 에서 확인하세요.
  논문에서 "FID 50k" 와 "FID 10k" 를 섞어 비교하면 안 되는 이유입니다.
- 사람 눈에 거의 같은 $\sigma=0.5$ px blur 나 $\sigma=5/255$ noise 가 FID 를 몇 배 올립니다. 두 조작 모두 precision 과 recall 을 함께 떨어뜨리고, 강한 blur ($\sigma=2$ px) 에서는 recall 이 거의 0 이 됩니다 — Inception 공간에서 blur 된 이미지는 실제 이미지 manifold 밖으로 통째로 이동합니다.
- 좌우 반전은 FID 를 거의 바꾸지 않고 (의미 보존), 5 개 class 만 남기면 같은 $N$ 에서 FID 가 크게 오르고 **recall** 이 떨어집니다 — mode dropping 신호.
- Bits/dim: 픽셀을 독립으로 보는 모델은 7–8 bpd 입니다. 참고로 논문 수치 (모두 CIFAR-10 test): PixelCNN++ 2.92 (Salimans et al., ICLR 2017), Glow 3.35 (Kingma & Dhariwal, NeurIPS 2018),
  DDPM ≤ 3.70 (Ho et al., NeurIPS 2020). 그 차이 4–5 bits/dim 이 **픽셀 사이 상관관계** 의 정보량입니다.

## 9. 생성 모델 family 지도

| Family | 학습 목적 | $\log p_\theta(x)$ | Sampling | 대표 | 이 코스 |
|---|---|---|---|---|---|
| Autoregressive | 정확한 MLE (chain rule) | 정확 | 순차 $D$ step | PixelCNN, GPT, VAR | §4, L03 (code prior), L13 |
| Latent variable / VAE | ELBO ($\le\log p$) | 하한 (IWAE 로 추정) | 1 pass | VAE, VQ-VAE, NVAE | §3, L02, L03 |
| Normalizing flow | 정확한 MLE (change of variables) | 정확 | 1 pass (가역) | RealNVP, Glow | §5 |
| GAN | adversarial (JS / Wasserstein) | 없음 | 1 pass | StyleGAN, GAN loss in VQGAN | §6, L03, L10 (ADD) |
| Diffusion / score | 가중 denoising = ELBO 의 변형 | ODE 로 정확 (비쌈) | 수십–수백 step | DDPM, EDM, SD | L04–L07, L09 |
| Flow matching | velocity regression | ODE 로 정확 | ODE step (적음) | Rectified flow, SD3, Qwen-Image | L08–L10 |

이번 강의의 실험이 보여 준 축: **likelihood 가능 여부, sampling 비용, mode coverage vs sample 품질**. Diffusion 과 flow matching 은
"VAE 의 안정적인 likelihood 학습 + GAN 수준의 sample 품질" 을 sampling step 수의 비용으로 얻은 family 입니다.

In [ ]:
RESULTS["runtime_min"] = rnd((time.time() - T_START) / 60, 2)
if device == "cuda":
    RESULTS["peak_mem_gb"] = rnd(torch.cuda.max_memory_allocated() / 1e9, 2)
save_results()
print(f"total runtime {RESULTS['runtime_min']:.1f} min, peak GPU memory {RESULTS.get('peak_mem_gb')} GB")
print("results ->", os.path.relpath(RES_PATH, HERE))

## Exercise

1. **Reverse KL 로 mixture 맞추기**: §2 의 $q$ 를 2-성분 Gaussian mixture 로 바꾸고 reverse KL 을 reparameterization 으로 최소화하세요 (성분 선택은 Gumbel-softmax 없이 $\mathbb E_q$ 를 성분별 기댓값의 가중합으로 정확히 쓰면 됩니다). 3-성분 target 에서 어떤 두 mode 를 고르나요? Forward KL (EM) 의 2-성분 해와 비교하세요.
2. **ELBO gap 직접 재기**: 학습된 GMM 에서 $q_\lambda(z)=(1-\lambda)\,p_\theta(z\mid x)+\lambda\,\mathrm{Uniform}(K)$ 로 두고 $\lambda\in[0,1]$ 에 대해 ELBO 와 $\log p_\theta(x)-\mathrm{KL}(q_\lambda\|p_\theta(z\mid x))$ 를 계산해 두 값이 같음을 확인하세요.
3. **AR 의 순서**: §4 의 모델을 $p(x_2)p(x_1\mid x_2)$ 순서로 다시 학습하세요. Test NLL 은 이론적으로 같아야 하는데 실제로도 같은가요? Swiss roll 에서 어느 순서가 더 쉬운지, 그 이유를 조건부 분포의 모양으로 설명하세요.
4. **Flow 의 위상 한계**: RealNVP 의 block 수를 2, 4, 8, 16 으로 바꿔 8 Gaussians 의 test NLL 과 mode 사이 "다리" 의 질량 (중심에서 $3\sigma$ 밖에 있는 sample 비율) 을 재세요.
5. **FID 와 사람의 판단**: CIFAR-10 test 10k 에 JPEG 압축 (quality 90, 50, 10) 을 적용해 FID, precision, recall 을 재고, 같은 FID 를 주는 blur 수준과 이미지를 나란히 놓아 보세요. 어느 쪽이 "더 나빠 보이나요"?
6. **WGAN-GP**: `train_gan(TRAIN["8gauss"], kind="wgan", lr=1e-4, n_critic=5, steps=20000, snap_every=500)` 으로 학습해 mode 수와 high-quality 비율의 궤적을 위 두 설정과 비교하세요. 좁은 mode 에서 gradient penalty 가 무엇을 어렵게 만드나요?